# Original submission notebook

This copy comes from the actual August 2024 submission. Code cells are unchanged. Saved outputs and execution counts were cleared, cell IDs were assigned, and the title attribution was shortened. The original input hash is in `../data/manifest.json`. The companion report retains the original results.

This is historical source material containing known calculation errors and environment-changing setup code. Use `../portfolio_construction.ipynb` for the maintained analysis and `../AMENDMENTS.md` for corrections. The original `utils.py` dependency is retained here. Data are held once in `../data/`.


# Final Project

**January 2024 Cohort**

Submission by:
the author credited in the repository README

August 2024

### Portfolio Construction
 

#### Outline of the Final Project Problem

This project aims to construct an optimally diversified trading portfolio focusing on short-term and intraday strategies. We utilize a data-driven approach, combining the Black-Litterman model and the Fama-French Five Factors model. Our project endeavors to address these challenges by employing strategic asset selection and weighting, aiming to navigate the semi-strong market efficiency and minimize the impact of financial 'noise'. Through this Jupyter Notebook, we present a detailed report that encapsulates our methodologies, analyses, and findings, contributing to the broader discourse on  portfolio construction and optimization in the context of contemporary market conditions.

**Expected Outcomes:**
1. External data sources for time series ETF prices and macroeconomic factors.
2. Development of a robust, factor-bearing portfolio that adapts to market changes and investor preferences.
3. Enhanced understanding of how macroeconomic factors and advanced machine learning techniques can inform and improve short-term and intraday trading strategies.
4. Insights into the practical application of the Black-Litterman model in conjunction with deep learning predictions for dynamic portfolio management.

Fama-French datasets retrieved from: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html

### Import Libraries

Required to install: 1) Pandas version 1.5.3 to prevent AttributeError: 'Series' object has no attribute 'append' in utilizing pandas_ta "mcgd.py", line 24, in mcgd  mcg_ds = close[:1].append(mcg_cell[1:].).

In [ ]:
import pydot
try:
    # Create a simple dot graph to test installed package
    graphs = pydot.Dot(graph_type='graph')  
    print("Graphviz is working.")
except Exception as e:
    print("Graphviz might not be installed or configured correctly.", str(e))

In [ ]:
# To use yahooquery for collecting market capitalization data
# pip install yahooquery --user

# Import necessary libraries for Deep Learning, Optimization, etc.
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning) 

import subprocess
import pkg_resources
from pathlib import Path
import datetime
from datetime import date

required_pkg = 'pandas'
required_version = '1.5.3'

# Function to check if the correct version of the package is installed
def is_installed(package, version):
    try:
        pkg_resources.require(f"{package}=={version}")
        return True
    except pkg_resources.VersionConflict:
        return False
    except pkg_resources.DistributionNotFound:
        return False

# Install package if not installed with correct version
if not is_installed(required_pkg, required_version):
    try:
        print(f"Installing {required_pkg}=={required_version}...")
        subprocess.check_call([f"pip", "install", f"{required_pkg}=={required_version}"])
    except Exception as e:
        print(f"Failed to install {required_pkg}=={required_version}: {e}")

# Basic data handling and array operations
import pandas as pd
import pandas_ta as ta
import talib
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML  

import yfinance as yf 
import os

from utils import * 

import traceback
import platform

from joblib import dump

import yahooquery 
import requests
from bs4 import BeautifulSoup

from statsmodels.tsa.stattools import adfuller, acf
import statsmodels.api as sm

from sklearn.metrics import mean_squared_error

from tabulate import tabulate 

from scipy.optimize import minimize    

import math

# Set the display format to suppress scientific notation
pd.set_option('display.float_format', lambda x: '%.3f' % x)
 

In [ ]:
# Helper functions
def convert_market_cap(market_cap_str):
    # Remove the dollar sign and commas
    cleaned_str = market_cap_str.replace('$', '').replace(',', '')
    
    # Initialize a multiplier based on the presence of 'M' or 'B'
    multiplier = 1
    if cleaned_str.endswith('M'):
        multiplier = 1000000
        cleaned_str = cleaned_str[:-1]  # Remove the 'M'
    elif cleaned_str.endswith('B'):
        multiplier = 1000000000
        cleaned_str = cleaned_str[:-1]  # Remove the 'B'
    
    try:
        # Convert the cleaned string to float
        value = float(cleaned_str)
        
        # Apply the multiplier based on the presence of 'M' or 'B'
        return value * multiplier
    except ValueError as e:
        print(f"Error converting market cap to float: {e}")
        return None
    
def nested_dict_pairs_iterator(dict_obj):
    ''' This function accepts a nested dictionary as argument
        and iterate over all values of nested dictionaries
    '''
    # Iterate over all key-value pairs of dict argument
    for key, value in dict_obj.items():
        # Check if value is of dict type
        if isinstance(value, dict):
            # If value is dict then iterate over all its values
            for pair in nested_dict_pairs_iterator(value):
                yield (key, *pair)
        else:
            # If value is not dict type then yield the value
            yield (key, value)
            
# Function to convert percentage strings back to float
def percent_string_to_float(s):
    return float(s.strip('%')) / 100

# Function for proper normalization
def normalize_weights(weights):
    total = np.sum(weights)
    if total != 0:
        return (weights / total) * 100
    return weights

def portfolio_metrics(weights, R, C, rf):
    """
    Calculate the portfolio's expected return, volatility (risk), and Sharpe ratio,
    with debug printout statements for verification.
    
    :param weights: Weights of the assets in the portfolio (1D array).
    :param R: Expected returns of the assets (1D array).
    :param C: Covariance matrix of the assets (2D array).
    :param rf: Risk-free rate.
    :return: Tuple containing the portfolio return, portfolio volatility, and Sharpe ratio.
    """
    weights = np.array(weights)
    R = np.array(R)
    C = np.array(C)
    
    # Debug printouts
    print("Weights:", weights)
    print("Expected Returns:", R)
    print("Covariance Matrix:\n", C)
    print("Risk-free Rate:", rf)
    
    port_return = np.dot(weights, R)
    print("Portfolio Return:", port_return)
    
    port_vol = np.sqrt(np.dot(weights.T, np.dot(C, weights)))
    print("Portfolio Volatility:", port_vol)
    
    sharpe_ratio = (port_return - rf) / port_vol
    print("Sharpe Ratio:", sharpe_ratio)
    
    return port_return, port_vol, sharpe_ratio

### Load Data - Define the Investment Universe

Load the 20 ETFs data from Yahoo! Finance using yfinance Python package and market capitalization data web-scrapped from Kavout.com website, diversifying across different asset classes, regions, and sectors. 


### Issues with yfinance and Nasdaq.com website

As we attempt to download market capitalization data using Yahoo! finance package, we encountered an error "Exception: yfinance failed to decrypt Yahoo data response". The issue was raised at https://github.com/ranaroussi/yfinance/issues/1407 but there is
no solution available to decrypt Yahoo data response at time of writing. 

Nasdaq website has implemented anti-scraping detection and relevant market capitalization data is hidden from the JSON response, even when Selenium was used. Some website will also track the User-Agent information, if we sent requests too many times with the same User-Agent, our IP will get banned.

Therefore, we shall use Kavout.com here instead for market capitalization data collection. 

In [ ]:
# Data Collection and Preparation
def get_risk_free_rate():    
    # Fetch data from 13 WEEK TREASURY BILL (^IRX)
    tnx = yf.Ticker('^IRX')
    # Get historical market data
    hist = tnx.history(period="5d")  
    return float(hist['Close'].iloc[-1]) / 100  # Convert to decimal

def get_etf_market_cap_from_kavout(etf_symbol):     
    # Construct the URL for the ETF's page on Kavout
    url = f"https://www.kavout.com/etfs/{etf_symbol.lower()}/vanguard-growth-etf"
    print(url)
    # Send a GET request to fetch the page content
    response = requests.get(url)
    
    # Check if the request was successful
    if response.status_code == 200:
        # Parse the HTML content
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the market cap value using the provided structure
        market_cap_tag = soup.find('span', text='Market Cap (AUM)')
        if market_cap_tag:
            market_cap_value_str = market_cap_tag.find_next('cite').text
            
            # Convert the market cap string to float and adjust for million
            market_cap_value = convert_market_cap(market_cap_value_str)
            print(etf_symbol, ':', market_cap_value)
            
            return market_cap_value
        else:
            print(response.status_code)
            return None
    else:
        print(response.status_code)
        return None

def collect_data(symbols, start_date, end_date, interval='1d', marketcap=True):
    data = {}
    
    for symbol in symbols:
        # Retrieve OHLCV for each symbol 
        ohlcv_data = yf.download(symbol, start=start_date, end=end_date, interval=interval)
        ohlcv_data.index = ohlcv_data.index.tz_localize(None) 
        
        if marketcap:
            # Retrieve market cap for each symbol
            market_cap = get_etf_market_cap_from_kavout(symbol)

            # Add market cap as a constant column to the DataFrame
            ohlcv_data['Market Cap'] = market_cap
        
        data[symbol] = ohlcv_data
    
    return data

### Issues with Kavout.com

In the event of connection timeout from retries error, we have to manually click on the first url ('https://www.kavout.com/etfs/spy/vanguard-growth-etf') to 'unlock' access to Kavout.com before the script can access the market capitalization data from the website.  

In [ ]:
# Define the ETFs representing the factors, 
# SPY will be used as the benchmark
etfs = ['SPY', 'VUG', 'VTV', 'SPHD', 'VB',  
        'MTUM', 'ACWI', 'EEM', 'VEA', 'EFA', 
        'XLF', 'XLK', 'XLE', 'IEO', 
        'SCHO', 'TLT', 'GLD', 'USO', 
        'IYR', 'VNQ', 'VIXY'] 

# Define SPY as the market proxy
market_ticker = 'SPY'  

#END_DATE = '2024-04-21'
END_DATE = date.today().strftime('%Y-%m-%d')

# Restricting samples to 3 years, according to project brief
# daily period data from 2020-01-01 to today's date will be used  
historical_data = collect_data(etfs, '2020-01-01', END_DATE) 

### Workaround on Issues with 3rd Party Websites and yfinance Package

On successful collection of data from Kavout.com and Yahoo! Finance, we store the data into a csv file for backup data retrieval so that each running of this code will not trigger web scrapping alerts on 3rd party websites for our IP address. 

In this code section, we commented out the part on saving to csv to be used when needed. 

In [ ]:
dataframes_long_format = []
etfs_df = pd.DataFrame()

for etf, df in historical_data.items():
    # Reset index to turn the date index into a regular column
    df = df.reset_index()
    # Add an 'ETF' column to identify the data
    df['ETF'] = etf
    # Append to the list
    dataframes_long_format.append(df)

# Concatenate all ETF DataFrames into a single long-format DataFrame
etfs_df = pd.concat(dataframes_long_format)

# Set a index for Date
etfs_df.set_index('Date', inplace=True)
etfs_df.to_csv('./data/efts_dataset.csv')

# Display the combined DataFrame
etfs_df.head(5)

In [ ]:
# remove the comment out if we were to read from csv
# etfs_df = pd.DataFrame()
if etfs_df.empty:
    etfs_df = pd.read_csv('./data/efts_dataset.csv')
    etfs_df.set_index('Date', inplace=True)

### Establish Market Equilibrium Returns using the Black-Litterman Model

The model recommends to adjust the expected market returns by incorporating the views (expectations) of the investor regarding the performance of the assets. This approach starts with the market equilibrium returns (typically derived from CAPM) and then modifies these returns based on the investor's specific views and confidence levels. This helps in creating a more personalized and more accurate forecast of expected returns.

#### Calculate Excess Returns

SPY is our market benchmark and we are interested in the excess returns over a risk-free rate, therefore we subtract the risk-free rate from the returns of each ETF, including SPY. 

To accurately determine the risk-free rate (RFR) we use the 13 WEEK TREASURY BILL (^IRX) via yahooquery to extract the latest available yield. This is the annualized RFR and so the risk-free rate in our  calculations involving daily returns, we have to adjust it to a daily rate to maintain consistency in the time frames of the data. 

In [ ]:
# Risk-free rate
annual_rfr = get_risk_free_rate()
print('Annual RFR: ', annual_rfr)
risk_free_rate = annual_rfr / 252
print('Daily RFR: ', risk_free_rate) 

In [ ]:
# Calculate daily returns from 'Adj Close'
etfs_df['Daily Returns'] = etfs_df['Adj Close'].pct_change()

# Calculate excess returns by subtracting the risk-free rate from the daily returns
etfs_df['Excess Returns'] = etfs_df['Daily Returns'] - risk_free_rate

# Display the DataFrame to check
etfs_df[['Adj Close', 'Daily Returns', 'Excess Returns']].head(5)

In [ ]:
nan_etf_rows = etfs_df[etfs_df['Daily Returns'].isna()]
print("Rows with NaN in the 'Daily Returns' column:")
print(nan_etf_rows)
etfs_df = etfs_df.fillna(0.0)

Extract 'SPY' column for benchmark use in applying optimization techniques. When we consider the Min Tracking Error optimization, SPY's returns serve as a benchmark to minimize the portfolio's deviation.

In [ ]:
# Extract 'SPY' column for benchmark use
spy_returns = etfs_df[etfs_df['ETF'] == market_ticker]['Daily Returns']
spy_returns = spy_returns.fillna(0.0) # for 1st row NaN

spy_benchmark = etfs_df[etfs_df['ETF'] == 'SPY'].copy()
spy_benchmark = spy_benchmark.dropna() 

# Check spy_benchmark
cprint(spy_benchmark)

In [ ]:
etfs_df.head(5)

In [ ]:
# Remove 'SPY' from etfs_df
etfs_df = etfs_df[etfs_df['ETF'] != 'SPY']

# Drop NaN from etfs_df after extracting spy_benchmark
etfs_df = etfs_df.dropna()

In [ ]:
# Remove 'SPY' from the list of ETFs if it exists
if market_ticker in etfs:
    etfs.remove(market_ticker)
    
# Print etfs list tp check
print(etfs)

# Check etfs_df columns if contain market_ticker
if market_ticker in etfs_df.columns:
    print(market_ticker, " is in the columns")

etfs_df.columns

In [ ]:
# Create a DataFrame to store asset returns
asset_returns = pd.DataFrame(index=etfs_df.index.unique())

# Calculate daily returns for each ETF
for ticker in etfs:
    ticker_data = etfs_df[etfs_df['ETF'] == ticker][['Daily Returns']]
    asset_returns[ticker] = ticker_data['Daily Returns']
    
# Drop any rows with NaN values
asset_returns = asset_returns.dropna()

# Display the DataFrame to check
print(asset_returns.head(5))

**Beta Data Collection for ETFs:**

Beta is a measure of the sensitivity of an asset's returns to the returns of the market. It indicates how much the asset's returns move relative to the market's returns. A beta greater than 1 indicates that the asset's returns are more volatile than the market, while a beta less than 1 indicates that the asset's returns are less volatile than the market.

We use Yahoo Finance! to get beta values using the yfinance library. For ETFs, the script specifically looks for the 3-year beta (beta3Year) because beta values are typically calculated over a 3-year period for such funds.

In [ ]:
# Initialize a dictionary to store the betas
betas = {}

# Loop through each ETF to get the beta
for etf in etfs: 
    ticker = yf.Ticker(etf)
    info = ticker.info
    
    # Use 'beta3Year' for ETFs, else set as 'N/A'
    beta = info.get('beta3Year', 'N/A')
    
    # Store in the betas dictionary
    betas[etf] = beta

# Display the betas dictionary
betas

### Capital Asset Pricing Model (CAPM) Portfolio Expected Returns

Given that our investment universe primarily consists of U.S. equities, we use SPY, which tracks the S&P 500, as the benchmark for calculating regression betas. The code first calculates beta values by performing linear regression on historical return data. This is done by regressing the returns of each ETF against the returns of the SPY index, which serves as a proxy for overall market performance in the U.S. This gives us a fixed or historical measure of how sensitive each ETF’s returns are to market movements (i.e. SPY).

Next, we calculate the returns of the global market index, represented by ACWI, to use as a proxy for global market performance. The regression betas calculated against SPY are then combined with the global market returns to estimate the CAPM expected returns for each ETF in our portfolio.

In [ ]:
# Verify the calculated betas using linear regression
regression_betas = {}
for ticker in etfs:
    X = sm.add_constant(spy_returns.loc[asset_returns.index])
    y = asset_returns[ticker]
    model = sm.OLS(y, X).fit()
    regression_betas[ticker] = model.params[1]  # The slope of the regression line

# Compare the betas
betas_df = pd.DataFrame({'Calculated Beta': betas, 'Regression Beta': regression_betas})
betas_df

In [ ]:
# Calculate the market return (average return of the market proxy)
market_return = spy_returns.mean() * 252  # Annualize the daily returns

# Calculate equilibrium returns using CAPM
equilibrium_returns = {}
for ticker in etfs:
    equilibrium_return = annual_rfr + betas[ticker] * (market_return - annual_rfr)
    equilibrium_returns[ticker] = equilibrium_return
    
# Display the equilibrium returns
equilibrium_returns_df = pd.DataFrame.from_dict(equilibrium_returns, orient='index', columns=['Equilibrium Return'])
print(equilibrium_returns_df)

### Calculating Annualized Expected Returns and Covariances

Using the ETFs historical daily data to compute annualized expected returns and covariances. The dataset, including SPY benchmark as part of the universe, aids in establishing a covariance matrix that includes the market's behavior.

Formula for annualizing compounded returns from daily returns would be the following where 252 represents the typical number of trading days in a year and this accounts for the compounding effect accurately over the year:

<center>$\text{Annualized Expected Return}= (1 + \text{mean daily return})^{252} − 1$</center>

In the assets_historical_returns_and_covariances() function, we calculate Compounded Returns which is more precise for longer-term investments or when the impact of daily return variability is significant. Then we calculate the daily returns' covariance and then multiply by 252 to annualize it, reflecting the variability in returns over a year.

We adjust the covariance matrix to allow the inclusion of any specific investor views or additional correlation information (adjusted_covariance). An additional correlation matrix will calculate a new covariance matrix by adjusting the original matrix based on the additional correlation data and the standard deviations of the returns.

In [ ]:
def assets_historical_returns_and_covariances(returns_df, additional_corr=None):
    """
    Calculate annualized historical returns and covariance matrix from a DataFrame, optionally 
    adjusted by an additional correlation matrix.

    :param returns_df: DataFrame with columns including 'Date', Asset Tickers, Daily Returns. Daily datasets.
    :param additional_corr: Optional DataFrame containing additional correlation data that might 
                            adjust the computed covariance matrix.
    :return: Annualized expected returns and annualized covariance matrix of returns for each ETF.
    """
    # Ensure returns_df is not empty and has the correct format
    if returns_df.empty:
        raise ValueError("The returns DataFrame is empty")
    if len(returns_df.shape) != 2:
        raise ValueError("The returns DataFrame must be 2-dimensional") 

    # Calculate expected returns (mean daily returns) and annualize them
    expreturns = np.mean(returns_df, axis=0)
    expreturns = (1 + expreturns) ** 252 - 1  # Annualize returns using compounded formula 

    # Calculate covariance matrix of returns and annualize
    covars = np.cov(returns_df, rowvar=False) * 252  # Annualize covariances 

    # If an additional correlation matrix is provided, adjust the covariance matrix
    if additional_corr is not None:
        std_devs = np.sqrt(np.diag(covars))
        adjusted_covars = additional_corr.multiply(np.outer(std_devs, std_devs))
        return expreturns, adjusted_covars
    else:
        return expreturns, covars

In [ ]:
cprint(asset_returns)

In [ ]:
# Calculate historical returns and covariances
R, C = assets_historical_returns_and_covariances(asset_returns)

In [ ]:
# Display the results
df_R = pd.DataFrame(R.round(3), columns=["Annualized Expected Returns"])

# Print the DataFrame nicely
print(df_R)

print("Shape of the covariance matrix:", C.shape)

print("\nAnnualized Covariance Matrix:") 

# Convert the NumPy array to a DataFrame with ETF names as index and columns
df_C = pd.DataFrame(C, index=etfs, columns=etfs)
 
# Style the DataFrame for a nicer display
df_C = df_C.round(3).style.set_table_styles(
    [{'selector': 'th', 'props': [('font-size', '7pt')]},
     {'selector': 'td', 'props': [('font-size', '7pt')]}]
)

# Display the styled DataFrame
display(df_C)

**Evaluation Comments:**

1. Expected Returns: The expected returns look plausible. For example, high beta and high volatility sectors like technology (XLK) typically have higher returns (0.239), while more stable investments like SCHO (short-term bonds) have lower returns (0.011).
2. Covariance Matrix: The covariance matrix shows the variances and covariances between the different assets. Diagonal elements represent the variances, and off-diagonal elements represent covariances. These values are also annualized.

### Systematic Backtesting 20 ETFs vs Market SPY

In this section, we conduct the calculation and visualization of the rolling beta and alpha for the 20 ETFs relative to the market (SPY) and perform systematic backtesting before engaging in portfolio optimization.

In [ ]:
# Align asset returns and SPY returns by their index (date)
asset_returns, spy_returns = asset_returns.align(spy_returns, join='inner', axis=0)

In [ ]:
spy_returns

In [ ]:
asset_returns

#### Period Window for Rolling Beta Calculation

The choice of the rolling window size (e.g., 30, 60, or 90 days) depends on several factors, such as:

1. Market Conditions: Shorter windows (e.g., 30 days) can be more sensitive to market changes, while longer windows (e.g., 90 days) provide smoother trends.
2. Asset Volatility: If the asset is highly volatile, a shorter window might capture changes in beta more effectively, while a longer window is more appropriate for less volatile assets.
3. Trading Frequency: The rolling window period should align with the investor's trading strategy. Shorter windows suit active traders, while longer windows fit long-term investors.
4. Data Availability: If limited data, using a shorter window might be necessary.

Let's choose the window size by using a performance metric such as Mean Squared Error (MSE), Akaike Information Criterion (AIC), or cross-validation to assess how well each rolling window captures the relationship between the asset and the benchmark.

Here’s our approach to automatically select the optimal rolling window size based on the Mean Squared Error (MSE) of the rolling regression:

In [ ]:
# Define possible rolling window sizes
window_sizes = [30, 60, 90]

# Function to compute rolling beta and alpha for a given window size
def compute_rolling_beta_alpha(asset_returns, spy_returns, window_size):
    betas = []
    alphas = []
    
    for i in range(window_size, len(asset_returns)):
        window_data = asset_returns.iloc[i - window_size:i]
        X = sm.add_constant(spy_returns.iloc[i - window_size:i])
        y = window_data
        
        if X.isnull().values.any() or y.isnull().values.any():
            betas.append(np.nan)
            alphas.append(np.nan)
        else:
            model = sm.OLS(y, X).fit()
            betas.append(model.params[1])  # Beta coefficient
            alphas.append(model.params[0])  # Alpha (intercept)
    
    return np.array(betas), np.array(alphas)

# Function to calculate the MSE for each window size
def evaluate_window_size(asset_returns, spy_returns, window_size):
    betas, _ = compute_rolling_beta_alpha(asset_returns, spy_returns, window_size)
    
    # Calculate MSE (Mean Squared Error) for the rolling beta estimates
    mse = mean_squared_error(spy_returns[window_size:], betas)
    return mse

# Find the optimal window size based on MSE
best_window = None
best_mse = float('inf')

for window_size in window_sizes:
    mse = evaluate_window_size(asset_returns['VUG'], spy_returns, window_size)
    print(f"Window Size: {window_size}, MSE: {mse}")
    
    if mse < best_mse:
        best_mse = mse
        best_window = window_size

print(f"Optimal Window Size: {best_window} with MSE: {best_mse}")

In [ ]:
# Define rolling window size
rolling_window = best_window

# Check for any NaN values
print("NaN values in asset_returns:", asset_returns.isna().sum())
print("NaN values in spy_returns:", spy_returns.isna().sum())

# Initialize DataFrames to store rolling beta and alpha for each ETF
rolling_betas = pd.DataFrame(index=asset_returns.index)
rolling_alphas = pd.DataFrame(index=asset_returns.index)

# Perform rolling regression for each ETF
for ticker in asset_returns.columns:
    betas = []
    alphas = []
    
    # Loop through data after the rolling window is available
    for i in range(rolling_window, len(asset_returns)):
        window_data = asset_returns.iloc[i-rolling_window:i]
        X = sm.add_constant(spy_returns.iloc[i-rolling_window:i])
        y = window_data[ticker]
        
        if X.isnull().values.any() or y.isnull().values.any():
            betas.append(np.nan)
            alphas.append(np.nan)
        else:
            model = sm.OLS(y, X).fit()
            betas.append(model.params[1])  # Beta coefficient
            alphas.append(model.params[0])  # Alpha (intercept)     
            
    # Store the rolling results
    rolling_betas[ticker] = pd.Series(betas, index=asset_returns.index[rolling_window:])
    rolling_alphas[ticker] = pd.Series(alphas, index=asset_returns.index[rolling_window:])


In [ ]:
# Display rolling betas and alphas
print("Rolling Betas:")
print(rolling_betas.tail())
print("\nRolling Alphas:")
print(rolling_alphas.tail())

In [ ]:
# Define a 5x4 grid for subplots (20 plots total) for Beta
fig_beta, axes_beta = plt.subplots(5, 4, figsize=(20, 15))  

# Define a 5x4 grid for subplots (20 plots total) for Alpha
fig_alpha, axes_alpha = plt.subplots(5, 4, figsize=(20, 15))  

# Flatten axes array for easier indexing
axes_beta = axes_beta.flatten()
axes_alpha = axes_alpha.flatten()

# Loop through all 20 ETFs to visualize their rolling beta and rolling alpha separately
for i, ticker in enumerate(asset_returns.columns[:20]):  # Visualizing all 20 ETFs
    # Plot rolling beta for each ETF in a subplot (beta plot)
    axes_beta[i].plot(rolling_betas.index, rolling_betas[ticker], label=f'{ticker} Beta', color='blue')
    axes_beta[i].set_title(f'Rolling Beta: {ticker}')
    axes_beta[i].set_xlabel('Date')
    axes_beta[i].set_ylabel('Beta')
    axes_beta[i].legend()
    
    # Plot rolling alpha for each ETF in a subplot (alpha plot)
    axes_alpha[i].plot(rolling_alphas.index, rolling_alphas[ticker], label=f'{ticker} Alpha', color='green')
    axes_alpha[i].set_title(f'Rolling Alpha: {ticker}')
    axes_alpha[i].set_xlabel('Date')
    axes_alpha[i].set_ylabel('Alpha')
    axes_alpha[i].legend()

# Adjust layout for better spacing on both figures
fig_beta.tight_layout()
fig_alpha.tight_layout()

# Display the plots
plt.show()


In this section, we compute key performance metrics and P&L returns for each ETF compared to the market (SPY). This includes calculating Sharpe Ratio, Sortino Ratio, Maximum Drawdown, Annualized Return, Volatility, and Cumulative P&L returns for each ETF.

Key metrics:
1. Sharpe Ratio: Risk-adjusted return, higher is better.
2. Sortino Ratio: Like Sharpe, but penalizes only downside risk, focusing on harmful volatility.
3. Maximum Drawdown: The worst peak-to-trough decline, lower is better.
4. Annual Return: The ETF's average return, annualized.
5. Volatility: The ETF's risk (standard deviation of returns), annualized.
6. Cumulative Returns (P&L): The total profit or loss of the ETF over time.

In [ ]:
# Function to calculate Sharpe Ratio
def calculate_sharpe_ratio(returns, rf_rate=0.01, periods_per_year=252):
    # Annualized return and volatility
    annual_return = returns.mean() * periods_per_year
    annual_volatility = returns.std() * np.sqrt(periods_per_year)
    # Sharpe ratio
    sharpe_ratio = (annual_return - rf_rate) / annual_volatility
    return sharpe_ratio, annual_return, annual_volatility

# Function to calculate Sortino Ratio
def calculate_sortino_ratio(returns, rf_rate=0.01, periods_per_year=252):
    # Calculate downside deviation (volatility of negative returns)
    negative_returns = returns[returns < 0]
    downside_deviation = np.sqrt(np.mean(negative_returns**2)) * np.sqrt(periods_per_year)
    # Annualized return
    annual_return = returns.mean() * periods_per_year
    # Sortino ratio
    sortino_ratio = (annual_return - rf_rate) / downside_deviation
    return sortino_ratio

# Function to calculate Maximum Drawdown
def calculate_max_drawdown(cumulative_returns):
    cumulative_high = cumulative_returns.cummax()
    drawdown = (cumulative_returns - cumulative_high) / cumulative_high
    max_drawdown = drawdown.min()
    return max_drawdown

# Function to calculate cumulative returns (P&L)
def calculate_cumulative_returns(returns):
    return (1 + returns).cumprod()

In [ ]:
# Initialize a DataFrame to store the results
performance_metrics = pd.DataFrame(columns=['ETF', 'Annual Return', 'Volatility', 'Sharpe Ratio', 'Sortino Ratio', 'Max Drawdown'])

# Loop through each ETF to calculate performance metrics
for ticker in asset_returns.columns:
    etf_returns = asset_returns[ticker]
    spy_returns = spy_returns
    
    # Calculate cumulative returns (P&L)
    etf_cumulative_returns = calculate_cumulative_returns(etf_returns)
    spy_cumulative_returns = calculate_cumulative_returns(spy_returns)
    
    # Calculate Sharpe ratio and other metrics
    sharpe_ratio, annual_return, volatility = calculate_sharpe_ratio(etf_returns)
    sortino_ratio = calculate_sortino_ratio(etf_returns)
    max_drawdown = calculate_max_drawdown(etf_cumulative_returns)
    
    # Store the metrics in the DataFrame
    performance_metrics = performance_metrics.append({
        'ETF': ticker,
        'Annual Return': annual_return,
        'Volatility': volatility,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino_ratio,
        'Max Drawdown': max_drawdown
    }, ignore_index=True) 

# Sort the performance_metrics DataFrame by multiple metrics with a priority list
performance_metrics = performance_metrics.sort_values(
    by=['Annual Return', 'Sharpe Ratio', 'Volatility', 'Sortino Ratio', 'Max Drawdown'],
    ascending=[False, False, True, False, True]
)


In [ ]:
# Plot cumulative P&L returns for ETFs vs SPY
plt.figure(figsize=(12, 8))

# Plot SPY cumulative returns
plt.plot(spy_cumulative_returns.index, spy_cumulative_returns, label='SPY', color='black', linewidth=2)

# Plot cumulative returns for each ETF
for ticker in asset_returns.columns:
    etf_cumulative_returns = calculate_cumulative_returns(asset_returns[ticker])
    plt.plot(etf_cumulative_returns.index, etf_cumulative_returns, label=ticker)

# Add plot title and labels
plt.title("Cumulative P&L Returns for ETFs vs SPY")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.legend()
plt.show()

In [ ]:
# Calculate performance metrics for SPY
spy_sharpe_ratio, spy_annual_return, spy_volatility = calculate_sharpe_ratio(spy_returns)
spy_sortino_ratio = calculate_sortino_ratio(spy_returns)
spy_max_drawdown = calculate_max_drawdown(spy_cumulative_returns)

# Add SPY metrics to the DataFrame for comparison
performance_metrics = performance_metrics.append({
    'ETF': 'SPY',
    'Annual Return': spy_annual_return,
    'Volatility': spy_volatility,
    'Sharpe Ratio': spy_sharpe_ratio,
    'Sortino Ratio': spy_sortino_ratio,
    'Max Drawdown': spy_max_drawdown
}, ignore_index=True)

# Display the full performance comparison including SPY
print("Performance Metrics Comparison (ETFs vs SPY):")
performance_metrics

**Evaluation Comments**

Comparisons of ETFs vs market SPY benchmark on **Annual Return**, **Volatility**, **Sharpe Ratio**, **Sortino Ratio**, and **Max Drawdown**:

| ETF  | Annual Return Comparison  | Volatility Comparison  | Sharpe Ratio Comparison  | Sortino Ratio Comparison | Max Drawdown Comparison  | Key Observations |
|------|---------------------------|------------------------|--------------------------|--------------------------|--------------------------|---------------------------|
| VB   | 3.9x higher than SPY       | 5x higher than SPY     | 17.5% lower than SPY      | 3.5x higher than SPY      | 25% deeper than SPY       | High returns and risk, strong Sharpe and Sortino, a top performer. |
| XLK  | 3x higher than SPY         | 2.6x higher than SPY   | 20% higher than SPY       | 2.5x higher than SPY      | 1x lower than SPY         | Excellent risk-adjusted performance, high Sharpe ratio. |
| TLT  | 1.9x higher than SPY       | 3.4x higher than SPY   | 41.4% lower than SPY      | 2.5x higher than SPY      | 44% deeper than SPY       | Defensive asset with low returns, high volatility, but decent downside risk management. |
| GLD  | 1.4x higher than SPY       | 1.3x higher than SPY   | 12.7% higher than SPY     | 2x higher than SPY        | 35% lower than SPY        | Defensive, low volatility, good risk-adjusted returns, portfolio stabilizer. |
| IEO  | Similar to SPY         | 2.2x higher than SPY   | 58.2% lower than SPY      | 4x lower than SPY         | 97% deeper than SPY       | High drawdown and poor Sharpe/Sortino ratios, risky energy sector exposure. |
| EFA  | 18.8% lower than SPY       | Similar to SPY         | 24.4% lower than SPY      | 16% lower than SPY        | Similar to SPY       | Low returns but stable, moderate risk-adjusted performance. |
| IYR  | 1.9x lower than SPY        | 1.2x higher than SPY   | 2.5x lower than SPY       | 2.5x lower than SPY       | 26% deeper than SPY       | Low returns with moderate volatility, but significant drawdown. |
| VEA  | 2.7x lower than SPY        | Similar to SPY         | 2.9x lower than SPY       | 3x lower than SPY         | Similar to SPY            | Low returns and risk-adjusted metrics, underperforms SPY. |
| VUG  | 3.4x lower than SPY        | 1.9x higher than SPY   | 7.7x lower than SPY       | 9.7x lower than SPY       | Similar to SPY       | Underperforms with low Sharpe and Sortino ratios. |
| MTUM | 4.5x lower than SPY        | 1.5x higher than SPY   | 9.1x lower than SPY       | 10.6x lower than SPY      | Similar to SPY       | Underperforms SPY with poor risk-adjusted metrics. |
| XLE  | 4.5x lower than SPY        | 2.4x higher than SPY   | 14.5x lower than SPY      | 17x lower than SPY        | 80% deeper than SPY       | Energy sector struggles, high volatility and drawdown. |
| VNQ  | 5x lower than SPY          | 1.3x higher than SPY   | 9.4x lower than SPY       | 9.8x lower than SPY       | 26% deeper than SPY       | Low returns with significant drawdowns, weak risk-adjusted metrics. |
| XLF  | Negative | 1.9x higher than SPY   | Negative | Negative | 27% deeper than SPY       | Underperforms with negative returns and poor risk-adjusted performance. |
| ACWI | Negative | 1.6x higher than SPY   | Negative | Negative | Similar to SPY            | Global ETF underperforms SPY with negative returns. |
| VIXY | Negative | 5x higher than SPY     | Negative | Negative | 2.9x deeper than SPY      | Extremely volatile, large drawdown, performs poorly in calm markets. |
| VTV  | Negative | 1.8x higher than SPY   | Negative | Negative | 9.2% deeper than SPY      | Value ETF underperforms SPY with negative risk-adjusted returns. |
| USO  | Negative | 2.4x higher than SPY   | Negative | Negative | 2.5x deeper than SPY      | Poor performance and very high drawdowns, extreme volatility in oil markets. |
| SPHD | Negative | 2x higher than SPY     | Negative | Negative | 23% deeper than SPY       | Poor performance, with significant downside risk and drawdown. |
| SCHO | Negative | Similar to SPY         | Negative | Negative | 83% lower than SPY        | Short-term bonds perform poorly with minimal risk-adjusted returns. |
| EEM  | Negative | 1.7x higher than SPY   | Negative | Negative | 18% deeper than SPY       | Emerging markets underperform, with high drawdown and poor risk-adjusted performance. |
| SPY  | 0.154	                | 0.215	             | 0.667	                 | 0.628	                 | -0.337                 | SPY is the benchmark for comparison. |


- **Top Performers**: **VB**, **XLK**, and **GLD** had significantly higher returns compared to SPY with decent or better risk-adjusted metrics.
- **Underperformers**: ETFs like **XLF**, **ACWI**, **VIXY**, **USO**, **VTV**, **SPHD**, and **EEM** showed negative returns, underperforming SPY across multiple metrics, often with higher volatility and deeper drawdowns.
- **Defensive Assets**: **GLD** and **TLT** performed better than SPY in terms of downside protection (Sortino ratio), although their returns were lower than SPY’s.
 

### Calculate ETFs Market Capitalisation 



In [ ]:
# Aggregate the latest market cap for each ETF
latest_caps = etfs_df.groupby('ETF')['Market Cap'].last()

# Calculate market weights from capitalizations
W = latest_caps / latest_caps.sum()

# Reindex W to match the order of etf_names
W = W.reindex(etfs)

print("Sum of Final Weights:", W.sum())
print("\nInitial Market Weights:")  # Initial weights before capping in next section 
W

**Evaluation Comments:**

1. In this section, we observe a high weight (> 10%) allocations to VEA, VTV and VUG which will skew the portfolio towards concentrated risk. 

2. Active Management Goals: Our aim to outperform the SPY benchmark using advanced techniques like deep learning forecasts and factor-based constructions may require a more dynamic or alternative weighting scheme.

3. Market cap weighting is generally more aligned with long-term, buy-and-hold strategies rather than short-term trading.

To align with industry best practices for weight allocations [1], we adjust the weights to ensure better diversification and reduce concentration risk. The following steps are taken:

1. Cap Individual ETF Weights: Ensure no single ETF exceeds 10% of the portfolio. 
2. Redistribute Excess Weight: Redistribute the excess weight proportionally to the remaining ETFs, maintaining the overall diversification.

**Reference:**

[1] https://www.spglobal.com/spdji/en/documents/methodologies/methodology-sp-us-style.pdf p10: "If any company's weight exceeds 10%, that company's weight is capped at 10% and all excess weight is proportionally redistributed to all uncapped companies within the index."

In [ ]:
# Make a copy from initial market weights
W_a = W.copy()

# Cap any individual ETF's weight at 10%
cap_threshold = 0.10

for ticker in W_a.index:
    print(f"{ticker} original weight: {W_a[ticker]:.4f}")
    if W_a[ticker] > cap_threshold:
        excess_weight = W_a[ticker] - cap_threshold
        W_a[ticker] = cap_threshold
        
        # Redistribute the excess weight proportionally among remaining ETFs
        remaining_etfs = W_a[W_a.index != ticker]
        W_a[remaining_etfs.index] += (excess_weight * remaining_etfs / remaining_etfs.sum())        
        
        print(f"New weight for {ticker}: {W_a[ticker]:.4f}, Excess weight: {excess_weight:.4f}")

print("Sum of Final Weights:", W_a.sum())

In [ ]:
# Updating the DataFrame
df_R_W = pd.DataFrame({'Return': R, 'Weight (based on market cap)': W, 'Adjusted Weight': W_a})

# Weights after adjusting with redistribution
print("Adjusted Market Weights:\n" )  
display(df_R_W)

### Correlation Analysis

In [ ]:
# 'C' is a square matrix with dimensions equal to the length of 'W'
df_C = pd.DataFrame(C, columns=W_a.index, index=W_a.index)

# Creating a heatmap and display
plt.figure(figsize=(15, 8))  # Set the figure size for better readability
sns.heatmap(df_C, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('ETF Correlations') 
plt.show()

**Evaluation Comments**

1. There is strong correlation between VB and TLT (0.75), VIXY and TLT (0.56), VB and XLK (0.61) and between VIXY and VB (0.66).

2. Negative correlations are between VIXY and XLE (-0.41), VIXY and VUG (-0.38), and VIXY and SPHD (-0.38). These negative correlations underscore its potential as a diversification tool during increased market volatility.

3. ETFs with correlations near zero, such as VEA with many others, suggest that these may provide diversification benefits when combined with other ETFs in the portfolio.

4. Low correlations between XLE (energy) and ETFs from other sectors or geographies can be exploited for reducing overall portfolio risk.

5. VIXY stands out as it has the most negative correlations with other ETFs, which is expected given its nature as a volatility ETF. 

6. GLD also behaves differently from most equity ETFs, reinforcing its role as a potential safe haven or hedge.

In [ ]:
# We use the following definition to filter out 'VB', 'VTV', 'TLT', 'VIXY', 'EEM', 
# 'VEA', 'EFA','XLF', 'SCHO', 'USO', 'IYR', 'VNQ' to reduce the number of ETFs to 8
# in line with narrowing down portfolio choice to < 10 ‘optimal bets’.
etf_names =  ['VUG', 'SPHD', 'MTUM', 'ACWI',  
              'XLK', 'XLE', 'IEO', 'GLD']

# Ensure df_C only includes the specific ETFs of interest
df_sub = df_C.loc[etf_names, etf_names]

# Convert the sub-matrix to a dictionary correctly
correlation_data = df_sub.to_dict()

# Style the DataFrame to reduce font size
styled_df = df_sub.round(3).style.set_table_styles(
    [{'selector': 'th', 'props': [('font-size', '8pt')]},
     {'selector': 'td', 'props': [('font-size', '8pt')]}]
)

# Display the styled DataFrame
display(styled_df)

In [ ]:
# Convert the dictionary to a DataFrame
correlation_df = pd.DataFrame(correlation_data) 

# Ensure asset_returns includes only the ETFs present in correlation_df
asset_returns_filtered = asset_returns[correlation_df.index]

# Adjusted R and C based on correlation_data
R, C = assets_historical_returns_and_covariances(asset_returns_filtered, correlation_df)

# Display the results
print("Annualized Expected Returns:\n", R)

print("Shape of the covariance matrix:", C.shape)

# Convert the NumPy array to a DataFrame with ETF names as index and columns
# Filtered list of etfs are in etf_names
df_C = pd.DataFrame(C, index=etf_names, columns=etf_names)
 
# Style the DataFrame for a nicer display
df_C = df_C.round(3).style.set_table_styles(
    [{'selector': 'th', 'props': [('font-size', '7.8pt')]},
     {'selector': 'td', 'props': [('font-size', '7.8pt')]}]
)

# Display the styled DataFrame
display(df_C)

**Role of the Covariance Matrix in Black-Litterman Model**

Risk Assessment:

1. The covariance matrix measures the covariance of returns between each pair of assets in the portfolio. It is fundamental in assessing the risk of the portfolio because it provides a measure of how assets move in relation to one another.

2. In the context of Black-Litterman, it helps in quantifying the risk associated with both the market equilibrium and any deviations from it due to the investor's views.

We filter $W$ and $W_a$ from 20 ETFs to select 8 ETFs for the next section.

In [ ]:
# Filter betas_df, W, and W_a to include only ETFs in etf_names
W = W.reindex(etf_names)

# Calculate the total allocation in the new W
total_allocation = W.sum()

# Calculate the redistribution amount
redistribution_amount = 1.0 - total_allocation

# Distribute the redistribution amount equally among the remaining ETFs
W += redistribution_amount / len(etf_names)

# Validate W total allocations is 100%
print('Sum of Final Weights:', W.sum())
W

In [ ]:
# Filter betas_df, W, and W_a to include only ETFs in etf_names
W_a = W_a.reindex(etf_names)

# Calculate the total allocation in the new W
total_allocation = W_a.sum()

# Calculate the redistribution amount
redistribution_amount = 1.0 - total_allocation

# Distribute the redistribution amount equally among the remaining ETFs
W_a += redistribution_amount / len(etf_names)

# Validate W_a total allocations is 100%
print('Sum of Final Weights:', W_a.sum())
W_a

We filter from 20 ETFs to select 8 ETFs for the next section in calculating the CAPM Portfolio Expected Returns relative to ACWI.

In [ ]:
betas_all_etfs_df = betas_df.copy() # for calculate the CAPM portfolio 20 ETFs' expected returns relative to ACWI
betas_df = betas_df.reindex(etf_names) # reduce down to 8 ETFs for Betting against Beta (BAB)
betas_df

## Implementing the Black-Litterman Model

This code defines the starting point for the Black-Litterman model by calculating the market equilibrium returns $Π$, which are adjusted later based on investor views and market outlook.

In [ ]:
# Black-Litterman Model

# Risk aversion coefficient (value taken from Idzorek (2007))
risk_aversion = 3.07

# Calculate implied market equilibrium returns (Π)
Pi = risk_aversion * np.dot(C, W_a)

# Convert to DataFrame for better readability
Pi_df = pd.DataFrame(pd.Series(Pi, index=W_a.index), columns=['Implied Market Returns'])
Pi_df

### Calculate CAPM Excess Returns

In [ ]:
# Calculate the CAPM excess returns relative to ACWI (a proxy for the global market)

# Download historical data for ETFs and a global market index (e.g., ACWI)
ACWI = historical_data['ACWI']['Adj Close']
ACWI

In [ ]:
etfs_df

In [ ]:
print(etfs_df['Daily Returns'].describe())
print(etfs_df[['ETF', 'Daily Returns']].head(20))
print(etfs_df.isna().sum())
print(etfs_df.groupby('ETF')['Adj Close'].describe())

In [ ]:
extreme_returns = etfs_df[etfs_df['Daily Returns'] > 2.5]   
print(extreme_returns) 

In [ ]:
# Calculate Historical Expected Returns  
prices_df = etfs_df.pivot_table(index='Date', columns='ETF', values='Adj Close')
returns = prices_df.pct_change().dropna() # Calculate daily returns
excess_returns = returns - risk_free_rate
historical_excess_returns = excess_returns.mean() * 252  # Annualize by multiplying by 252 trading days 
historical_excess_returns = historical_excess_returns.reindex(etf_names)
historical_excess_returns

In [ ]:
# Calculate proxy for global market portfolio, e.g. ACWI
ACWI_returns = ACWI.pct_change().dropna()
global_market_return = ACWI_returns - risk_free_rate # Proxy for global market portfolio
global_market_return

In [ ]:
# Calculate the CAPM Portfolio Expected Returns relative to ACWI, annualize it
capm_portfolio_returns = betas_all_etfs_df['Regression Beta'] * global_market_return.mean() * 252
capm_portfolio_returns

In [ ]:
# Implied Equilibrium Return Vector Π
implied_equilibrium_returns = equilibrium_returns_df['Equilibrium Return']

# Reindex and filter all series to match the order and ETFs in etf_names
capm_portfolio_returns = capm_portfolio_returns.reindex(etf_names)
implied_equilibrium_returns = implied_equilibrium_returns.reindex(etf_names)

# Combine the results into a DataFrame
table_1 = pd.DataFrame({
    'Asset Class': etf_names,
    'Historical μ_Hist': historical_excess_returns.values,
    'CAPM Portfolio μ_P': capm_portfolio_returns.values,
    'Implied Equilibrium Return Vector Π': implied_equilibrium_returns.values
})

# Define the relative asset class descriptions
asset_classes = {
    'VUG': 'US Large Growth', 
    'SPHD': 'US High Dividend',
    'VB': 'US Small Cap',
    'MTUM': 'US Momentum',
    'ACWI': 'Global Equity',
    'EEM': 'Emerging Markets',
    'VEA': 'Developed Markets',
    'EFA': 'International Developed',
    'XLF': 'US Financials',
    'XLK': 'US Technology',
    'XLE': 'US Energy',
    'IEO': 'Oil & Gas Exploration',
    'SCHO': 'US Short-Term Bonds', 
    'GLD': 'Gold',
    'USO': 'Crude Oil',
    'IYR': 'US Real Estate',
    'VNQ': 'US REITs', 
    'Weighted Average' : 'Weighted Average',
    'Standard Deviation' : 'Standard Deviation',
    'High' : 'High',
    'Low' : 'Low'    
}

# Calculate Weighted Average, Standard Deviation, High, and Low
weighted_avg = table_1[['Historical μ_Hist', 'CAPM Portfolio μ_P', 'Implied Equilibrium Return Vector Π']].mean()
std_dev = table_1[['Historical μ_Hist', 'CAPM Portfolio μ_P', 'Implied Equilibrium Return Vector Π']].std()
high_values = table_1[['Historical μ_Hist', 'CAPM Portfolio μ_P', 'Implied Equilibrium Return Vector Π']].max()
low_values = table_1[['Historical μ_Hist', 'CAPM Portfolio μ_P', 'Implied Equilibrium Return Vector Π']].min()

# Append these statistics to the table
table_1.loc['Weighted Average'] = ['Weighted Average'] + list(weighted_avg)
table_1.loc['Standard Deviation'] = ['Standard Deviation'] + list(std_dev)
table_1.loc['High'] = ['High'] + list(high_values)
table_1.loc['Low'] = ['Low'] + list(low_values)

# Add the asset class descriptions to the DataFrame
table_1['Asset Class Description'] = table_1['Asset Class'].map(asset_classes)

# Format the values to three decimal places with a % sign
table_1 = table_1.applymap(lambda x: f"{x*100:.2f}%" if isinstance(x, (int, float)) else x)

# Reorder the columns to have the description first
table_1 = table_1[['Asset Class', 'Asset Class Description'] + table_1.columns.tolist()[1:-1]]

# Reset the index to ensure it's numeric
table_1 = table_1.reset_index(drop=True)

# Display Table 1
table_1

**Historical Returns (μ_Hist):**

1. Performance Disparity: The historical returns show variability across asset classes, with XLK (US Technology) achieving the highest return at 18.89%, while SPHD (US High Dividend) lags behind at 3.91%. The overall portfolio’s historical weighted average return stands at 11.63%, showcasing strong performance from growth and momentum assets such as VUG (US Large Growth) and MTUM (US Momentum), alongside cyclical sectors like XLE (US Energy) and IEO (Oil & Gas Exploration).

**CAPM Portfolio Returns (μ_P):**

2. Risk-Adjusted Returns: The CAPM portfolio returns are generally more conservative, with XLK still leading at 8.29%. However, GLD (Gold) shows significantly lower CAPM returns at 0.75%, indicating a potential lack of risk premium despite its historical appeal as a safe haven. The overall weighted average return under CAPM stands at 6.31%, highlighting a shift towards a more risk-averse outlook compared to historical performance.

**Implied Equilibrium Returns (Π):**

3. Optimistic Market Expectations: The Black-Litterman model projects notably higher expected returns across most asset classes, with VUG and MTUM showing implied returns of 17.54% and 15.13%, respectively. ACWI (Global Equity) and XLK continue to present a bullish outlook, with implied returns of 15.65% and 18.07%. On the other hand, GLD sees a more tempered expectation at 6.20%, indicating that market sentiment does not favor gold as strongly going forward. The overall weighted average of implied returns is 13.66%, reflecting the model’s optimism.

**Others:**

4. Volatility and Risk: The standard deviation across the historical, CAPM, and Black-Litterman implied returns shows a decrease in risk estimates, with the implied returns’ standard deviation at 3.84%, compared to 5.90% for historical returns. This highlights the stabilizing influence of the Black-Litterman model on portfolio volatility.

In [ ]:
# Calculating the optimum weights using the adjusted weights from df_R_W

# Reindex and filter to match the order and ETFs in etf_names
df_R_W = df_R_W.reindex(etf_names)

# risk_aversion = 3.07  # defined risk aversion coefficient above
w_hist = df_R_W['Return'] * (1 / risk_aversion)  # Simplified as w = μ / λ for each asset
w_hist = (w_hist / w_hist.sum()) * 100  # Normalize to sum to 100%
w_hist

In [ ]:
# Convert the relevant columns back to float
table_1['Historical μ_Hist'] = table_1['Historical μ_Hist'].apply(percent_string_to_float)
table_1['Implied Equilibrium Return Vector Π'] = table_1['Implied Equilibrium Return Vector Π'].apply(percent_string_to_float)
table_1

In [ ]:
# Provided return vectors from Table 1
mu_hist = table_1.iloc[:, int(table_1.columns.get_loc('Historical μ_Hist'))].values  
mu_hist = mu_hist[:len(etf_names)] # retain 16 ETFs return vectors
mu_hist

In [ ]:
# Normalize the weights
w_hist = normalize_weights(mu_hist)

# Debug: Check normalized weights
print("Normalized w_hist:", w_hist)

**Calculate Weights Based on Implied Equilibrium Return Vector Π:**

1. Using the implied equilibrium returns (from Π, i.e. Pi) to calculate the corresponding portfolio weights.
2. The weights are derived using the formula:

$𝑤_{implied}= {Π_𝑖 / {∑Π_𝑖}}$

3. Normalize these weights to ensure they sum to 100%.

In [ ]:
# Normalize the implied equilibrium return vector Π
w_implied = normalize_weights(Pi)

# Debug: Check normalized weights
print("Normalized w_implied:", w_implied)

In [ ]:
# Convert the asset_classes dictionary to a list based on the order of ETFs
asset_class_descriptions = [asset_classes[ticker] for ticker in table_1['Asset Class'][:len(etf_names)].values]

# Combine into the final Table 2 DataFrame 
table_2 = pd.DataFrame({
    'Asset Class': asset_class_descriptions,
    'Weight Based on Historical w_Hist': w_hist,
    'Weight Based on Implied Equilibrium Return Vector Π': w_implied,
    'Market Capitalization Weight w_mkt': W * 100,
    'Market Capitalization Weight (adjusted)': W_a * 100
})
        
# Format the weights to three decimal places with a % sign
table_2.iloc[:, 1:] = table_2.iloc[:, 1:].applymap(lambda x: f"{x:.3f}%")

# Display Table 2 using tabulate in HTML format
table_html_2 = tabulate(table_2, headers='keys', tablefmt='html', numalign='right', stralign='center') 
HTML(f'<div style="font-size:10px">{table_html_2}</div>')

**Evaluation Comments**

1. The portfolio generated from the Historical Return Vector results in highly concentrated and extreme allocations, which is not surprising given that this method heavily favors assets with strong past performance like XLK (Technology) and IEO (Oil & Gas). This highlights the inherent risk of relying solely on historical data, as it can lead to unrealistic and unbalanced portfolio allocations.

2. The Implied Equilibrium Return Vector portfolio shows substantial deviations from the benchmark market capitalization-weighted portfolio. This is especially evident in the allocation to sectors like SPHD (High Dividend) and XLE (Energy) while reducing exposure to XLK (Technology) and GLD (Gold). The implied returns suggest a dramatically higher allocation compared to the market capitalization weights. This discrepancy underscores how market expectations can diverge from historical performance.
 
3. A notable shift is the bearish outlook on GLD, with a negative allocation, contrasting with its historical role as a safe-haven asset.

4. Both historical and implied return methods overweight the Energy sector, reflecting both past performance and strong market expectations for the future.

5. The adjusted market cap weights provide a more diversified and moderate allocation, avoiding the extreme positions seen in the historical return portfolio.


### Determine Views to the Equilibrium Returns 

Integration of Views:

1. The BL model uniquely allows the incorporation of investor views on returns, which can be absolute or relative. These views are expressed as adjustments to the expected returns derived from the market model.

2. The covariance matrix is essential in this step because it helps to scale and modify these views based on the risk (volatility) associated with each asset. The views are adjusted by the uncertainty in these views, which is often derived or influenced by the elements of the covariance matrix.

Confidence in Views:

3. Investor confidence in each view affects how much these views will shift the final expected returns from the market equilibrium. The BL model uses a parameter called the uncertainty matrix (often a function of the covariance matrix) to scale these views based on confidence levels.

#### Macroeconomic Factors - Integration into Black-Litterman Model

In addition to traditional inputs like equilibrium returns and investor views, the Black-Litterman model can be enhanced by incorporating macroeconomic indicators, which provide insights into broader market trends and conditions. Specifically, the following four indicators can be integrated as "views" into the model which asset management firm Fullerton uses as part of their proprietary framework and methodology, utilized to generate insights into market dynamics and asset allocation decisions:

1. Global Growth: This indicator, derived from a combination of major global indices such as the S&P 500, Nikkei 225, and Euro Stoxx 50, provides a normalized measure of economic growth across key regions. It captures the overall economic momentum, which can influence expected returns.

2. Global Inflation: Utilizing Gold prices as a proxy and inflation data from the World Bank’s Prospects Group represented by metrics like the Median, Average, and GDP-weighted Average inflation rates, offers insights into the inflationary environment. Inflation expectations can impact asset prices and returns, as higher inflation often leads to higher interest rates, affecting bond yields and equity valuations.

3. Global Liquidity: Measured by indicators such as the 10-Year Treasury Yield and data from the Bank for International Settlements (BIS) global liquidity indicators (GLIs), this reflects the availability of liquidity in the financial system. High liquidity typically supports asset prices, while tightening liquidity conditions can lead to increased volatility and lower valuations.

4. Risk Appetite: Using the VIX index as a proxy, this indicator gauges market sentiment and risk tolerance. A higher VIX indicates greater market volatility and risk aversion, which can lead to changes in asset allocation preferences.

The section below contains the code detailing the computation of these four factors using data from January 1996 to August 2024:

In [ ]:
def normalize_series(series, scale_factor=3):
    """ Normalize a Pandas Series to a mean of 0 and a specified range. """
    return (series - series.mean()) / series.std() * scale_factor

# Define the tickers for global growth
growth_indices = {
    'S&P 500': '^GSPC',
    'Nikkei 225': '^N225',
    'Hang Seng': '^HSI',
    'Shanghai Composite': '000001.SS',
    'Euro Stoxx 50': '^STOXX50E',
    'Nifty 50': '^NSEI'
}

# Other indicators
inflation_ticker = 'GLD'  # Gold ETF as a proxy for inflation
liquidity_ticker = '^TNX'  # 10 Year Treasury Yield as a proxy for liquidity
risk_appetite_ticker = '^VIX'  # VIX for risk appetite

# Download and normalize historical data
start_date = '1996-01-01'
end_date = '2024-08-22'
normalized_data_growth = {}

for name, ticker in growth_indices.items():
    data = yf.download(ticker, start=start_date, end=end_date)['Close']
    normalized_data_growth[name] = normalize_series(data)

# Combined global growth data
combined_growth_data = pd.DataFrame(normalized_data_growth).mean(axis=1)

# Download and normalize data for other indicators
inflation_data = yf.download(inflation_ticker, start=start_date, end=end_date)['Close']
normalized_inflation_data = normalize_series(inflation_data)

liquidity_data = yf.download(liquidity_ticker, start=start_date, end=end_date)['Close']
normalized_liquidity_data = normalize_series(liquidity_data)

risk_appetite_data = yf.download(risk_appetite_ticker, start=start_date, end=end_date)['Close']
normalized_risk_appetite_data = normalize_series(risk_appetite_data)

# Plotting
plt.figure(figsize=(14, 10))

# Global Growth
plt.subplot(2, 2, 1)
plt.plot(combined_growth_data, label='Normalized Combined Global Growth')
plt.title('Normalized Combined Global Growth')
plt.legend()

# Global Inflation
plt.subplot(2, 2, 2)
plt.plot(normalized_inflation_data, label='Normalized Global Inflation (Gold)')
plt.title('Normalized Global Inflation')
plt.legend()

# Global Liquidity
plt.subplot(2, 2, 3)
plt.plot(normalized_liquidity_data, label='Normalized Global Liquidity (10 Year Treasury Yield)')
plt.title('Normalized Global Liquidity')
plt.legend()

# Risk Appetite
plt.subplot(2, 2, 4)
plt.plot(normalized_risk_appetite_data, label='Normalized Risk Appetite (VIX)')
plt.title('Normalized Risk Appetite')
plt.legend()

plt.tight_layout()
plt.show()

**Normalized Combined Global Growth:**

1. This plot shows the normalized global economic growth trends from 1996 to 2024. This indicator, derived from a combination of major global indices such as the S&P 500, Nikkei 225, and Euro Stoxx 50, provides a normalized measure of economic growth across key regions. The data indicates periods of strong growth (e.g., early 2000s, post-2008 recovery) and downturns (e.g., 2008 financial crisis, COVID-19 pandemic). The recent trend shows a sharp increase, suggesting strong economic recovery or growth. 

**Normalized Global Inflation (Gold Prices):**

2. The inflation plot uses Gold prices as a proxy. There is a noticeable increase in inflation starting around 2008, with a peak in 2022-2024. This trend aligns with recent concerns about rising inflation globally, possibly due to factors like supply chain disruptions and increased demand post-pandemic.

**Normalized Global Liquidity (10Y Treasury Yield):**

3. This plot, using the 10-Year Treasury Yield, shows periods of high and low liquidity. High liquidity is seen during economic crises (e.g., 2008, 2020), where central banks often lower interest rates to stimulate the economy. Recent trends indicate fluctuating liquidity, reflecting uncertainty in monetary policy.

**Normalized Risk Appetite (VIX):**

4. The VIX index plot represents market volatility and investor risk appetite. Peaks correspond to market turmoil (e.g., 2008 crisis, 2020 pandemic). The plot suggests that risk appetite is cyclical, with periods of heightened fear followed by more stable periods.

The four graphs show a comprehensive view of the macroeconomic environment, highlighting the global growth, inflation, liquidity, and risk appetite over a time series data.

#### Data Collection for Global Inflation (Alternative to Gold)

The World Bank’s Prospects Group has constructed a global database of inflation dated April 2024 version. The database covers up to 209 countries over the period 1970-2023, and includes six measures of inflation in three frequencies (annual, quarterly, and monthly) for Headline consumer price index (CPI) inflation.

We extract the Global headline consumer price index (HCPI) inflation data from the excel file downloaded from [2] as an alternative to using Gold prices in our previous step. This field provides a comprehensive measure of inflation by tracking the changes in the price level of a basket of consumer goods and services across the globe, making it an ideal indicator to plot a global inflation trends.

References:
[2] https://www.worldbank.org/en/research/brief/inflation-database

In [ ]:
# Load the specific sheet 'Aggregate' and use the provided range of data from the Excel file
file_path = './data/Inflation-data.xlsx'   

# Read the specified range from the 'Aggregate' sheet
inflation_data_agg = pd.read_excel(file_path, sheet_name='Aggregate', usecols='B:BG', nrows=5)
inflation_data_agg

In [ ]:
# Transpose the data for easier manipulation
inflation_data_transposed = inflation_data_agg.transpose()

# Set the first row as the header after transposing
new_header = inflation_data_transposed.iloc[0]
inflation_data_transposed = inflation_data_transposed[1:]
inflation_data_transposed.columns = new_header

# Remove the non-year rows
inflation_data_transposed = inflation_data_transposed.drop(['Description', 'Aggregation', 'Sample country'], axis=0)

# Parse the years as integers and reset index
inflation_data_transposed.index = pd.to_numeric(inflation_data_transposed.index).astype(int)
inflation_data_transposed = inflation_data_transposed.reset_index()

# Dropping HCPI_GLOBAL_MED_BAL ('Median (balanced group)') for clearer plot display
inflation_data_transposed = inflation_data_transposed.drop('HCPI_GLOBAL_MED_BAL', axis=1)

inflation_data_transposed.tail(5)

In [ ]:
# Rename the columns to have clear names
inflation_data_transposed.columns = ['Year', 'Median', 'Average', 'GDP-weighted Average'] #, 'Median (balanced group)']
inflation_data_transposed.tail(5)

In [ ]:
# Select the range from 1996 to 2023
inflation_data_selected_years = inflation_data_transposed[inflation_data_transposed['Year'].between(1996, 2023)]

# Plot the inflation data
plt.figure(figsize=(14, 7))
for column in inflation_data_selected_years.columns[1:]:
    plt.plot(inflation_data_selected_years['Year'], inflation_data_selected_years[column], label=column)

plt.title('Global Inflation Rates (1996 - 2023)')
plt.xlabel('Year')
plt.ylabel('Inflation Rate (%)')
plt.legend()
plt.grid(True)
plt.show()

#### Data Collection for Global Liquidity (Alternative to 10Y Treasury Yield)

To create the Global Liquidity plot, we used data from the Bank for International Settlements (BIS) global liquidity indicators (GLIs), specifically the file "WS_GLI_csv_col.csv." [3], instead of using the 10 Year Treasury Yield as proxy. 

The BIS defines "global liquidity" as the ease of financing in global financial markets. The GLIs track credit to non-bank borrowers, covering both bank loans and funding from global bond markets through international debt securities (IDS). The focus is on foreign currency credit denominated in major reserve currencies (US dollars, euros, Japanese yen) to non-residents.

The code below filters the excel data to isolate specific conditions (i.e. All currencies, all countries, all sectors, banks total lending), normalizes the selected data, and then visualizes it as a bar chart to represent normalized global liquidity trends.

Reference:
[3] https://data.bis.org/topics/GLI

In [ ]:
# Load the CSV file
file_path = './data/WS_GLI_csv_col.csv'   
data = pd.read_csv(file_path)
data.tail(5)

In [ ]:
# Create a condition that checks data pulling criteria: All currencies, all countries, all sectors, Banks-total.
condition = (data['L_POS_TYPE'] == 'I') & (data['BORROWERS_CTY'] == '5J') & (data['BORROWERS_SECTOR'] == 'A') & (data['UNIT_MEASURE'] == '770') 

# Use the condition to filter the DataFrame
global_liquidity_data = data[condition]  
global_liquidity_data

In [ ]:
# Filtering columns to keep only the years data
years_columns = [col for col in global_liquidity_data.columns if '-Q' in col]
global_liquidity_data = global_liquidity_data[years_columns]
global_liquidity_data.tail

In [ ]:
# Extracting the data for the two rows (153 and 154) and normalizing based on their own mean 
data_row = global_liquidity_data.loc[142, '1996-Q1':'2023-Q2']
 
normalized_data_row = (data_row - data_row.mean()) / data_row.std() 

# Plotting row 143 as a bar chart
plt.figure(figsize=(15, 7))
plt.bar(normalized_data_row.index, normalized_data_row, label="Row 143")
plt.axhline(y=0, color='gray', linestyle='--')
plt.xlabel('Quarter')
plt.ylabel('Normalized Value')
plt.title('Normalized Global Liquidity Data (Row 143)')
#plt.xticks(rotation=45)

# Customizing x-ticks to show only years
# Data starts from '1996-Q1' to '2023-Q2'
years = np.arange(1996, 2024)  # From 1996 to 2023
# Only select quarters that represent the start of each year, e.g., '1996-Q1'
year_ticks = [f'{year}-Q1' for year in years]

# Set custom ticks and labels
plt.xticks(ticks=year_ticks, labels=years, rotation=45)

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
global_liquidity_data.dropna()

columns_to_keep = [col for col in global_liquidity_data.columns if col > '1995-Q4']

global_liquidity_data = global_liquidity_data[columns_to_keep]
global_liquidity_data.tail(5)

In [ ]:
# Plotting
plt.figure(figsize=(14, 10))

# Global Growth
plt.subplot(2, 2, 1)
plt.plot(combined_growth_data, label='Normalized Combined Global Growth')
plt.title('Global Growth')
plt.legend()

# Global Inflation - New Plot
plt.subplot(2, 2, 2)
inflation_data_selected_years = inflation_data_selected_years[inflation_data_selected_years['Year'].between(1996, 2023)]
for column in inflation_data_selected_years.columns[1:]:
    plt.plot(inflation_data_selected_years['Year'], inflation_data_selected_years[column], label=column)
plt.title('Global Inflation Rates')
plt.xlabel('Year')
plt.ylabel('Inflation Rate (%)')
plt.legend()
plt.grid(True)
 
# Normalized Global Liquidity Data (Row 143)
plt.subplot(2, 2, 3)  
data_row = global_liquidity_data.loc[142, '1996-Q1':'2023-Q2']
normalized_data_row = (data_row - data_row.mean()) / data_row.std() 
plt.bar(normalized_data_row.index, normalized_data_row, label="Global Liquidity (Banks Lending; All Countries)")
plt.axhline(y=0, color='gray', linestyle='--')
plt.xlabel('Quarter')
plt.ylabel('')
plt.title('Global Liquidity')
# Customizing x-ticks to show only years
years = np.arange(1996, 2024)
# Only select quarters that represent the start of each year, e.g., '1996-Q1'
year_ticks = [f'{year}-Q1' for year in years]
# Set custom ticks and labels
plt.xticks(ticks=year_ticks, labels=years, rotation=45)
plt.legend()
plt.grid(True)

# Risk Appetite
plt.subplot(2, 2, 4)
plt.plot(normalized_risk_appetite_data, label='Normalized Risk Appetite (VIX)')
plt.title('Risk Appetite')
plt.legend()

plt.tight_layout()
plt.show()

#### Generation of Q and P Matrices

To incorporate investor perspectives into the Black-Litterman (BL) model for the eight selected ETFs, we integrate macroeconomic factors such as inflation expectations, liquidity conditions, and the broader economic outlook. This integration enables a customized portfolio optimization process.

1. Q Matrix (Views Vector): This matrix quantifies the investor's specific views on the relative performance of assets. Each element in Q represents the expected outperformance of one asset over another in a given view. For example, if an investor expects ETF1 to outperform ETF2, this will be reflected in the corresponding Q element.

2. P Matrix (Link Matrix): This matrix links each investor view to the relevant assets. For a view where ETF1 is expected to outperform ETF2, +1 is assigned to ETF1’s column and -1 to ETF2’s column. This approach ensures the model understands which assets are involved in each view and the direction of expected outperformance or underperformance.

After constructing the P and Q matrices, the BL model combines these investor views with the market's implied equilibrium returns (Π) and the covariance matrix of asset returns. The model adjusts for the uncertainty in views (Ω) and applies a scaling factor (τ) to account for the influence of market equilibrium.

In [ ]:
def create_views_and_link_matrix(etf_names, investor_views):
    """
    Creates the Q (views) matrix and P (link) matrix based on investor views.
    
    :param etf_names: A list of ETF names (strings).
    :param investor_views: A list of tuples representing the investor's views.
                           Each tuple is in the format: (ETF1, Relation, ETF2, View Value),
                           where 'Relation' is '>' if ETF1 is expected to outperform ETF2,
                           and '<' if ETF1 is expected to underperform ETF2.
    :return: A tuple containing the Q matrix (as a NumPy array) and the P matrix (as a NumPy array).
    """
    num_views = len(investor_views)
    num_etfs = len(etf_names)
    Q = np.array([view[3] for view in investor_views])  # Extract the view values
    P = np.zeros((num_views, num_etfs))

    # Create a mapping from ETF names to their index positions for easy access
    name_to_index = {name: index for index, name in enumerate(etf_names)}

    # Populate the P matrix based on the views
    for i, view in enumerate(investor_views):
        etf1, relation, etf2, _ = view
        P[i, name_to_index[etf1]] = 1 if relation == '>' else -1
        P[i, name_to_index[etf2]] = -1 if relation == '>' else 1

    return Q, P

In [ ]:
# Print the ETF names (after correlation filtering) and state the investor views
print(etf_names)

investor_views = [
    ('GLD', '>', 'MTUM', 0.15),  
    ('XLK', '>', 'XLE', 0.08),
    ('SPHD', '>', 'VUG', 0.06)
]

# Generate the Q and P matrices
Q, P = create_views_and_link_matrix(etf_names, investor_views)

# Display the matrices
print('Views Matrix (Q):')
print(pd.DataFrame({'Views': Q}))

### Rationale on Investor's Views (as of August 22, 2024)

The views matrix Q indicate the investor's subjective views on the **expected excess returns** of specific assets or groups of assets:

- View 1: expects GLD to outperform by 15% relative to MTUM.  
- View 2: expects XLK to outperform by 8% relative to XLE.
- View 3: expects SPHD to outperform by 6% relative to VUG.

**1. Gold (GLD) vs. Momentum (MTUM) Stocks:**

Given the high inflation environment, economic uncertainties and the backdrop in escalation of war in the Middle East, **gold is expected to outperform momentum stocks** as a hedge against high inflation. On the 3rd of Agust 2024, Iran and their allied groups vow to hit back at Israel in response to the killing of Hamas leader by Israel, while Washington increases military presence in the Middle East.  

**2. Technology (XLK) vs. Energy (XLE):**

Despite experiencing high volatility, the Technology sector is anticipated to outperform the Energy sector. This expectation is driven by strong underlying fundamentals and continuous innovation, even as commodity prices may stabilize and the initial excitement around AI cools. 

The long-term potential of Generative AI remains promising, and Technology companies continue to report robust earnings. As a result, we maintain a **positive outlook on the Technology sector**.

**3. High Dividend (SPHD) vs. Growth Stocks (VUG):**

With a high-interest rate environment, **high dividend stocks may offer a more stable income stream** compared to growth stocks, of which growth stocks could be more sensitive to rising rates, and the escalation of war in the Middle East. U.S. stocks are expected to gradually increase as cost pressures stabilize and corporate earnings start to recover. 

We are seeing recovering trend particularly in Asia and Europe. In the U.S., a positive surprise in earnings could propel the market further. Although U.S. inflation is decreasing at a slower pace than anticipated, the general expectation remains that central banks in developed markets will begin lowering interest rates later this year.

In [ ]:
# Create the DataFrame for the Link Matrix (P)
link_matrix_df = pd.DataFrame(P, columns=etf_names)

# Format the DataFrame with one decimal place and convert it to a list of lists for tabulate
link_matrix_list = link_matrix_df.applymap(lambda x: f'{x:.1f}').values.tolist()

# Display the Link Matrix (P) as html
table_html = tabulate(link_matrix_list, headers=etf_names, tablefmt='html', numalign='right', stralign='center')
print('\nLink Matrix (P):')
HTML(f'<div style="font-size:10px">{table_html}</div>')

**View 1 (Row 0):**

This view suggests that GLD (Gold) is expected to outperform MTUM (Momentum). The +1 in GLD's column and -1 in MTUM's column indicate a positive view on GLD relative to MTUM.  

**View 2 (Row 1):**

This view suggests that XLK (Technology) is expected to outperform XLE (Energy). The +1 in XLK's column and -1 in XLE's column indicate a positive view on XLK relative to XLE.

**View 3 (Row 2):**

This view suggests that SPHD (High Dividend Low Volatility) is expected to outperform VUG (Growth). The +1 in SPHD's column and -1 in VUG's column indicate a positive view on SPHD relative to VUG.

In [ ]:
# Check on the covariance matrix
C

**Implied Equilibrium Return Vector (Π):**

The vector Π represents the market's consensus expected returns under the assumption of equilibrium. We derived this from the market portfolio's composition and the risk aversion coefficient in the Reverse Optimization section. 

Code Reference: In the code, Pi represents this implied equilibrium return vector. This is the starting point for the Black-Litterman model and reflects the market's neutral expectations before considering the investor's views.

In [ ]:
# Check on the implied market equilibrium returns
Pi

### Integrating Market Equilibrium and Investor Views with Uncertainty and Scaling

After generating the P (Link Matrix) and Q (Views Vector) matrices, the next step in the Black-Litterman (BL) model is to integrate these matrices with the equilibrium market returns and the covariance matrix of the asset returns.

The Black-Litterman model combines the implied market equilibrium returns (Π) with the adjusted views (through Q and P), weighted by the uncertainty in the views (Ω) and the scalar (τ). Earlier in computing Π, we use the risk aversion coefficient (λ) of the U.S. market at roughly 3.07, taken from the work of Idzorek (2007).  This combination creates the new expected return vector called the BL_return_vector in the code, which reflects both market equilibrium and subjective views. The formula is stated as:

<center>BL_return_vector$= Π + σ_{scaled} ⋅ P^T ⋅ (P ⋅ σ_{scaled} ⋅ P^T + ω)^{−1} ⋅ (Q − P ⋅ Π)$</center>

The scalar (τ) and uncertainty matrix (Ω) determine the relative influence of the implied equilibrium returns and the investor's views on the final adjusted returns. A higher τ or lower Ω value gives more weight to the views, while a lower τ or higher Ω value places more emphasis on the implied equilibrium returns. 

In the BL model, the covariance matrix is used to minimize the portfolio variance for a given level of expected return N×N matrix, where N is the number of assets or factors in the portfolio. This is at the heart of mean-variance optimization. In our case, N=20 corresponding to the 20 ETFs defined in the investment universe. 

The Link Matrix is a K×N matrix, where K is the number of views, and N is the number of assets or factors. While the Views Vector is Kx1 vector containing the expected reeturns specified by investor's views. 

**Code Implementation**

The following code implements the Black-Litterman model to adjust the returns and covariance matrix based on investor views and then performs portfolio optimization. The confidence level of the views shall be set from 70-90%. 

We define omega as a diagonal matrix representing a specific confidence level from 70%-90% for each view, instead of a scalar matrix applying 70% confidence to all views. If there are multiple views, omega should have a value corresponding to the confidence for each view on its diagonal.

In [ ]:
# Confidence levels for each view (0 < confidence <= 1)
confidence_levels = [0.90, # escalation of war in the Middle East  
                     0.70, # positive outlook on tech compared to energy
                     0.80] # high interest rate to continue and growth stocks are sensitive to it

# Calculate the uncertainty matrix omega (inverse of confidence levels)
omega = np.diag([1 / c for c in confidence_levels])

# Scaling factor (tau): measure of uncertainty of the prior estimate of the mean returns
tau = 0.1
sigma_scaled = tau * C

In [ ]:
temp = Pi + sigma_scaled @ P.T @ np.linalg.inv(P @ sigma_scaled @ P.T + omega) @ (Q - P @ Pi)

print("Adjusted Returns:\n", temp)

In [ ]:
# Black-Litterman formula to adjust returns  

# Step 1: Calculate P @ sigma_scaled @ P.T
P_sigma_Pt = P @ sigma_scaled @ P.T
print("Shape of P @ sigma_scaled @ P.T:", P_sigma_Pt.shape)

# Step 2: Add omega to the result and invert
inv_term = np.linalg.inv(P_sigma_Pt + omega)
print("Shape of inv(P @ sigma_scaled @ P.T + omega):", inv_term.shape)

# Step 3: Calculate Q - P @ Pi
Q_minus_Pi = Q - P @ Pi
print("Shape of Q - P @ Pi:", Q_minus_Pi.shape)

# Step 4: Calculate sigma_scaled @ P.T @ inv_term
sigma_Pt_inv = sigma_scaled @ P.T @ inv_term
print("Shape of sigma_scaled @ P.T @ inv_term:", sigma_Pt_inv.shape)

# Step 5: Final step: sigma_Pt_inv @ (Q - P @ Pi)
final_term = sigma_Pt_inv @ Q_minus_Pi
print("Shape of final term (sigma_scaled @ P.T @ inv_term @ (Q - P @ Pi)):", final_term.shape)

# Step 6: Calculate the adjusted returns
adjusted_returns = Pi + final_term
print("Shape of adjusted returns:", adjusted_returns.shape)

print("Adjusted Returns:\n", adjusted_returns)

In [ ]:
# Ensure all variables are NumPy arrays
C = np.array(C)  # Covariance matrix
sigma_scaled = np.array(sigma_scaled)
P = np.array(P)
omega = np.array(omega)

# Adjusted covariance matrix - if we account for the effect of views on the overall risk structure of the portfolio. 
adjusted_covariance = (
    C 
    + sigma_scaled 
    - sigma_scaled @ P.T @ np.linalg.inv(P @ sigma_scaled @ P.T + omega) @ P @ sigma_scaled
)

print("Adjusted Covariance Matrix:\n")   

# Convert the NumPy array to a DataFrame with ETF names as index and columns
df_C = pd.DataFrame(adjusted_covariance, index=etf_names, columns=etf_names)
 
# Style the DataFrame for a nicer display
df_C = df_C.round(3).style.set_table_styles(
    [{'selector': 'th', 'props': [('font-size', '7.8pt')]},
     {'selector': 'td', 'props': [('font-size', '7.8pt')]}]
)

# Display the styled DataFrame
display(df_C)

In [ ]:
# Black-Litterman formula to calculate adjusted returns (mu_implied)
mu_implied_calculated = Pi + sigma_scaled @ P.T @ np.linalg.inv(P @ sigma_scaled @ P.T + omega) @ (Q - P @ Pi)

# Step 2: Compare Calculated vs. Provided Adjusted Returns
comparison = pd.DataFrame({
    'Adjusted Returns': adjusted_returns,
    'Calculated Adjusted Returns': mu_implied_calculated
})

# Calculate the difference
comparison['Difference'] = comparison['Adjusted Returns'] - comparison['Calculated Adjusted Returns']

# Step 3: Display the comparison table
print(comparison)

# Check if the differences are within an acceptable tolerance
tolerance = 1e-6  # Define an acceptable tolerance level
validation_result = np.all(np.abs(comparison['Difference']) < tolerance)

print("Validation Result:", "Pass" if validation_result else "Fail")


In [ ]:
# Step 1: Calculate the adjustment term
adjustment_term = sigma_scaled @ P.T @ np.linalg.inv(P @ sigma_scaled @ P.T + omega) @ P @ sigma_scaled

# Step 2: Calculate the adjusted covariance matrix
adjusted_covariance_calculated = C + sigma_scaled - adjustment_term

# Step 3: Compare the matrices
difference = adjusted_covariance_calculated - adjusted_covariance
max_difference = np.max(np.abs(difference)) 

# Check if the differences are within an acceptable tolerance
tolerance = 1e-6
validation_result = np.all(np.abs(difference) < tolerance)

print("Validation Result:", "Pass" if validation_result else "Fail")

### Calculating the New Combined Return Vector

In the Black-Litterman model, the New Combined Return Vector (often denoted as 𝐸[𝑅]) is the result of combining the market equilibrium returns (Π) with the investor's views.

This adjusted return vector $μ_{implied}$ or 𝐸[𝑅] is what the model recommends as the expected returns after incorporating both market consensus and the investor’s specific views.

In [ ]:
# Use adjusted_returns to calculate the new combined return vector weights
w_implied_new = normalize_weights(adjusted_returns)
print("Normalized w_implied:\n", w_implied_new)

In [ ]:
# Combine into the final Table 2 DataFrame 
table_2 = pd.DataFrame({
    'Asset Class': asset_class_descriptions,
    'Weight Based on Historical w_Hist': w_hist,
    'Weight Based on Implied Equilibrium Return Vector Π': w_implied,
    'Weight Based on New Combined Return Vector Π': w_implied_new,
    'Market Capitalization Weight w_mkt': W * 100, # Convert to percentage weights
    'Market Capitalization Weight (adjusted)': W_a * 100 # Convert to percentage weights
})
        
# Format the weights to two decimal places with a % sign
table_2.iloc[:, 1:] = table_2.iloc[:, 1:].applymap(lambda x: f"{x:.2f}%")

# Display Table 2 using tabulate in HTML format
table_html_2 = tabulate(table_2, headers='keys', tablefmt='html', numalign='right', stralign='center') 
HTML(f'<div style="font-size:10px">{table_html_2}</div>')

**Black-Litterman Posterior Returns Calculation:**

This part of the code calculates the Black-Litterman return vector (BL_return_vector), which represents the adjusted expected returns after incorporating investor views.

- Pi: Implied equilibrium returns (prior returns).
- sigma_scaled: Covariance matrix (adjusted - tau x C). The term $σ_{scaled}$ is used instead of τΣ where $σ_{scaled}$ represents the scaled covariance matrix of returns. 
- The scaling factor tau (τ) is often applied directly to Σ (C) to reflect the investor's confidence in the market equilibrium returns.
- P: Matrix representing investor views.
- Q: Expected returns according to the investor views.
- omega: Uncertainty matrix for investor views.

This formula integrates the market equilibrium returns and investor views using Bayesian updating to compute the posterior return vector.

In [ ]:
# Black-Litterman return vector
BL_return_vector = Pi + sigma_scaled.dot(P.T).dot(np.linalg.inv(P.dot(sigma_scaled).dot(P.T) + omega).dot(Q - P.dot(Pi)))

print("Black-Litterman Return Vector:\n")
BL_return_vector

In quantitative finance, especially when working with complex models like the Black-Litterman model, it's essential to ensure that the results from different stages of the computation are consistent. 

The comparison of adjusted_returns (computed using the Black-Litterman formula) and BL_return_vector (computed in a potentially different or complementary manner) computed in above can serve as a validation step here.

In [ ]:
# Compare the adjusted_returns with the BL_return_vector
comparison = np.allclose(adjusted_returns, BL_return_vector)

# Output the result of the comparison
if comparison:
    print("Adjusted Returns and Black-Litterman Return Vector are consistent.")
else:
    print("Discrepancy found: Further investigation required.")


Now, we use the formula to calculate the adjusted expected returns **$μ_{implied}$** that reflect both the market consensus and the investor’s views, weighted by confidence levels:

<br>
<center>$μ_{implied}$ $= Π + τ.Σ.P^T (P.τ.Σ.P^T + Ω)^{−1} . (Q−P.Π)$</center>
<br>
where $(P.τ.Σ.P^T + Ω)^{−1}$ is the term that adjusts the investor's views for their uncertainty, relative to the market's uncertainty.

For the value of Tau (τ) in the equation above, we adhere to the precedent established by Pyo & Lee (2018) and set it to 0.1 accordingly.

Π is calculated as Π = risk_aversion × np.dot (𝐶,𝑊𝑎), using the adjusted weights 𝑊𝑎 rather than the raw market capitalization weights 𝑊.

$μ_{implied}$ : Π directly influences the weight allocation, reflecting the market equilibrium adjusted by the investor's views. Best used when the goal is to determine how the market's implied returns should theoretically influence portfolio allocation, assuming the investor's views are fully incorporated.


In [ ]:
# Scaling factor (tau): measure of uncertainty of the prior estimate of the mean returns
tau = 0.1

# Calculate the adjusted implied returns based on views
mu_implied = Pi + tau * np.dot(C, np.dot(P.T, np.linalg.inv(np.dot(P, np.dot(tau * C, P.T)) + omega) @ (Q - np.dot(P, Pi))))

# Calculate the weights from adjusted implied returns 
w_hat = normalize_weights(mu_implied)

# Calculate the differences
diff_return = mu_implied - Pi # mu_implied - Implied Equilibrium Return Vector, these are differences in return values, not percentages
diff_weight = w_hat - (W * 100) # These are differences in weights (should be in percentage terms)

# Create the table
table_3 = pd.DataFrame({
    'Asset Class': etf_names,
    'New Combined Return Vector E[R]': mu_implied , # in return values
    'Implied Equilibrium Return Vector Π': Pi , # in return values
    'Difference E[R] − Π': diff_return, # in return values
    'New Weight wˆ': w_hat, # in percentage weights
    'Market Capitalization Weight w_mkt': W * 100, # in percentage weights
    'Difference wˆ − w_mkt': diff_weight # in percentage weight differences
})

# Format the columns as needed
table_3.iloc[:, 4:] = table_3.iloc[:, 4:].applymap(lambda x: f"{x:.2f}%")

# Display the table
table_3

### Optimization of the Portfolio

The portfolio optimization process fine-tunes asset weights to achieve an optimal risk-return balance, leveraging insights from both market equilibrium and the investor's specific views. In this section, we use the adjusted returns from the Black-Litterman model (BL_return_vector) to guide our optimization strategy. 

Our approach focuses on minimizing tracking error relative to the benchmark portfolio (SPY) while also incorporating a robust backtesting framework to evaluate the portfolio's performance against key market factors.


### 1. Mean-Variance Portfolio

Mean-Variance Optimization is a fundamental concept in modern portfolio theory, introduced by Harry Markowitz in 1952. It provides a mathematical framework for constructing portfolios that maximize expected returns for a given level of risk (variance) or minimize risk for a given expected return. This method is widely used in portfolio management to achieve the most efficient allocation of assets, balancing risk and reward.

**Code Implementation**

We use Python's scipy.optimize.minimize function to find the optimal portfolio weights that minimize the portfolio's variance (risk) based on a given covariance matrix of asset returns. Result is optimal asset allocation that minimizes portfolio risk given the assets' covariance.

In [ ]:
# Define the objective function: minimize portfolio variance
def portfolio_variance(weights, cov_matrix):
    return weights.T @ cov_matrix @ weights  # Quadratic portfolio variance

# Define the constraint: weights must sum to 1 (fully invested)
def weight_sum_constraint(weights):
    return np.sum(weights) - 1

num_assets = len(etf_names)

# Bounds for the weights: no short-selling (weights between 0 and 1)
bounds = [(0, 1) for _ in range(num_assets)]

# Constraints for the optimization: fully invested portfolio
constraints = {'type': 'eq', 'fun': weight_sum_constraint}

# Perform the optimization to minimize variance
result = minimize(portfolio_variance, W_a, args=(C,), method='SLSQP',
                  bounds=bounds, constraints=constraints)

# Extract the optimal weights
mv_weights = result.x

# Combine ETF names with their corresponding weights
mv_weights_df = pd.DataFrame({'ETF': etf_names, 'MV Weight': mv_weights})

# Display the weights in a readable format
print("Mean-Variance Portfolio Weights:")
mv_weights_df 

In [ ]:
# Reuse previously computed annual risk-free rate
annual_rfr

In [ ]:
# Calculate the expected portfolio return
mv_return = np.dot(mv_weights, adjusted_returns)

# Calculate portfolio volatility (standard deviation)
mv_sharpe_vol = np.sqrt(np.dot(mv_weights.T, np.dot(C, mv_weights))) 

# Calculate Sharpe Ratio
mv_sharpe_ratio = (mv_return - annual_rfr) / mv_sharpe_vol

# Display the portfolio's performance
print(f"Mean-Variance Portfolio: Return = {mv_return:.2%}, Volatility = {mv_sharpe_vol:.2%}, Sharpe Ratio = {mv_sharpe_ratio:.2f}")

### 2. Maximum Sharpe Ratio Optimization

**Code Implementation**

The neg_sharpe_ratio function calculates the portfolio weights that maximize the Sharpe ratio, a key measure of risk-adjusted return. The Sharpe ratio represents the excess return per unit of risk, considering the difference between the portfolio's return and a specified risk-free rate. The function takes as inputs the portfolio weights, expected returns, covariance matrix, and the risk-free rate, and returns the negative Sharpe ratio to be minimized (since optimization routines minimize by default).

In [ ]:
# Portfolio Optimization: Maximum Sharpe Ratio Portfolio
def max_sharpe_ratio_portfolio(mu, S):
    num_assets = len(mu)
    inv_S = np.linalg.inv(S)
    ones = np.ones(num_assets)
    w_max_sharpe = inv_S @ mu / (ones.T @ inv_S @ mu)
    return w_max_sharpe

# Calculate the portfolios
w_max_sharpe = max_sharpe_ratio_portfolio(adjusted_returns, adjusted_covariance)

# Combine ETF names with their corresponding weights
w_max_sharpe_df = pd.DataFrame({'ETF': etf_names, 'Max Sharpe Weight': w_max_sharpe})

print("Max Sharpe Portfolio:")
print(w_max_sharpe_df)

In [ ]:
# Define the covariance matrix and initial weights from the Max Sharpe Portfolio
cov_matrix = adjusted_covariance  # The covariance matrix of asset returns
initial_weights = w_max_sharpe

# Objective function: minimize portfolio variance
def portfolio_variance(weights, cov_matrix):
    return np.dot(weights.T, np.dot(cov_matrix, weights))

# Constraint: fully invested portfolio (weights sum to 1)
def weight_sum_constraint(weights):
    return np.sum(weights) - 1

# Bounds for the weights: no short-selling, weights between 0 and 1
bounds = [(0, 1) for _ in range(len(initial_weights))]

# Constraints: fully invested portfolio
constraints = {'type': 'eq', 'fun': weight_sum_constraint}

# Perform quadratic optimization to minimize portfolio variance
result = minimize(portfolio_variance, initial_weights, args=(cov_matrix,), method='SLSQP',
                  bounds=bounds, constraints=constraints)

# Extract the optimized weights
optimal_weights = result.x 

# combine the optimized weights with the ETF names to display the final optimized portfolio 
max_sharpe_weights_df = pd.DataFrame({'ETF': etf_names, 'Optimized Weights': optimal_weights})

# Display the optimized portfolio
print("Optimized Portfolio Weights:") 
print(max_sharpe_weights_df)

We calculate the key performance metrics (returns, volatilities, and Sharpe ratios) for different portfolios. 


In [ ]:
# Calculate portfolio for the optimization strategies 
max_sharpe_return = np.dot(w_max_sharpe, adjusted_returns) 

# Volatility (standard deviations) for the portfolio 
max_sharpe_vol = np.sqrt(np.dot(w_max_sharpe.T, np.dot(adjusted_covariance, w_max_sharpe))) 

# Sharpe Ratio for the portfolio 
max_sharpe_sharpe = (max_sharpe_return - annual_rfr) / max_sharpe_vol if max_sharpe_vol != 0 else float('nan') 

In [ ]:
# Display the Max Sharpe initial portfolio
print(w_max_sharpe_df)

# Display the portfolio's performance
print(f"Max Sharpe Ratio Portfolio: Return = {max_sharpe_return:.2%},\n Volatility = {max_sharpe_vol:.2%}, Sharpe Ratio = {max_sharpe_sharpe:.2f}")

In [ ]:
# Calculate portfolio for the optimization strategies 
max_sharpe_return = np.dot(optimal_weights, adjusted_returns) 

# Volatility (standard deviations) for the portfolio 
max_sharpe_vol = np.sqrt(np.dot(optimal_weights.T, np.dot(adjusted_covariance, optimal_weights))) 

# Sharpe Ratio for the portfolio 
max_sharpe_sharpe = (max_sharpe_return - annual_rfr) / max_sharpe_vol if max_sharpe_vol != 0 else float('nan') 

# Display the optimized portfolio
print(max_sharpe_weights_df)

# Display the portfolio's performance
print(f"Max Sharpe Ratio Optimized Portfolio: Return = {max_sharpe_return:.2%},\n Volatility = {max_sharpe_vol:.2%}, Sharpe Ratio = {max_sharpe_sharpe:.2f}")

### 3. Higher-order Moments

In this section, we implement higher-order moments such as minimizing coskewness and maximizing cokurtosis:

1. Coskewness: This measures the relationship between an asset’s returns and the portfolio’s returns. A portfolio with positive coskewness tends to have higher potential for positive extreme returns, and negative coskewness indicates higher downside risk.

2. Cokurtosis: This refers to the relationship between an asset’s returns and the portfolio’s return distribution in terms of "fat tails" or extreme events. Higher cokurtosis indicates a greater likelihood of extreme events.

In portfolio optimization, traditional mean-variance optimization doesn't account for these higher-order moments. We will modify the portfolio optimization problem by adding coskewness and cokurtosis to the objective function.

Reference: https://stackoverflow.com/questions/41890870/how-to-calculate-coskew-and-cokurtosis

In [ ]:
asset_returns = asset_returns[etf_names]
asset_returns

In [ ]:
# Function to calculate cokurtosis
def cokurt(df, bias=False, fisher=True, variant='middle'):
    v = df.values
    s1 = sigma = v.std(0, keepdims=True)  # Standard deviation
    means = v.mean(0, keepdims=True)      # Mean of returns

    # Center the data
    v1 = v - means
    s2 = sigma ** 2
    s3 = sigma ** 3

    v2 = v1 ** 2
    v3 = v1 ** 3
    m = v.shape[0]

    # Compute 'left', 'right', or 'middle' variant of cokurtosis
    if variant in ['left', 'right']:
        kurt = pd.DataFrame(v3.T.dot(v1) / s3.T.dot(s1) / m, df.columns, df.columns)
        if variant == 'right':
            kurt = kurt.T
    elif variant == 'middle':
        kurt = pd.DataFrame(v2.T.dot(v2) / s2.T.dot(s2) / m, df.columns, df.columns)

    # Apply bias correction if necessary
    if not bias:
        kurt = kurt * (m ** 2 - 1) / (m - 2) / (m - 3) - 3 * (m - 1) ** 2 / (m - 2) / (m - 3)

    # Adjust for Fisher's definition of kurtosis (subtract 3 for normality)
    if not fisher:
        kurt += 3

    return kurt

# Function to calculate coskewness
def coskew(df, bias=False):
    v = df.values
    s1 = sigma = v.std(0, keepdims=True)  # Standard deviation of columns (assets)
    means = v.mean(0, keepdims=True)      # Mean of columns (assets)

    # Center the data by subtracting the means
    v1 = v - means
    s2 = sigma ** 2  # Variance

    # Calculate squared centered values (variance)
    v2 = v1 ** 2
    m = v.shape[0]  # Number of observations

    # Compute coskew matrix
    skew = pd.DataFrame(v2.T.dot(v1) / s2.T.dot(s1) / m, df.columns, df.columns)

    # Adjust for bias if necessary
    if not bias:
        skew *= ((m - 1) * m) ** 0.5 / (m - 2)

    return skew

# Coskewness Calculation (without np.einsum)
def calculate_portfolio_coskewness(weights, coskew_matrix):
    portfolio_coskewness = 0
    num_assets = len(weights)
    for i in range(num_assets):
        for j in range(num_assets):
            for k in range(num_assets):
                portfolio_coskewness += weights[i] * weights[j] * weights[k] * coskew_matrix[i, j, k]
    return portfolio_coskewness

# Cokurtosis Calculation (without np.einsum)
def calculate_portfolio_cokurtosis(weights, cokurt_matrix):
    portfolio_cokurtosis = 0
    num_assets = len(weights)
    for i in range(num_assets):
        for j in range(num_assets):
            for k in range(num_assets):
                for l in range(num_assets):
                    portfolio_cokurtosis += weights[i] * weights[j] * weights[k] * weights[l] * cokurt_matrix[i, j, k, l]
    return portfolio_cokurtosis

# Utility function that incorporates coskewness and cokurtosis
def utility_function(weights, expected_returns, cov_matrix, coskew_matrix, cokurt_matrix, risk_aversion):
    # Expected portfolio return
    portfolio_return = np.dot(weights, expected_returns)
    
    # Portfolio variance
    portfolio_variance = np.dot(weights.T, np.dot(cov_matrix, weights))
    
    # Portfolio coskewness
    portfolio_coskewness = calculate_portfolio_coskewness(weights, coskew_matrix)
    
    # Portfolio cokurtosis
    portfolio_cokurtosis = calculate_portfolio_cokurtosis(weights, cokurt_matrix)

    # Utility function combining return, variance, skewness, and kurtosis
    return portfolio_return - risk_aversion * portfolio_variance + portfolio_coskewness - portfolio_cokurtosis

# Function to optimize portfolio based on the utility function
def optimize_portfolio(expected_returns, cov_matrix, coskew_matrix, cokurt_matrix, risk_aversion):
    num_assets = len(expected_returns)
    
    # Initial equal weights
    initial_weights = np.ones(num_assets) / num_assets
    
    # Bounds for the weights (between 0 and 1, no short selling)
    bounds = [(0, 1) for _ in range(num_assets)]
    
    # Constraint: weights must sum to 1
    constraints = ({'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1})
    
    # Objective function: maximize utility
    result = minimize(lambda weights: -utility_function(weights, expected_returns, cov_matrix, coskew_matrix, cokurt_matrix, risk_aversion),
                      initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
    
    return result.x 

# Calculate the Coskewness Matrix
def calculate_coskew_matrix(returns):
    num_assets = returns.shape[1]
    coskew_matrix = np.zeros((num_assets, num_assets, num_assets))
    means = returns.mean(axis=0)
    std_devs = returns.std(axis=0)
    
    for i in range(num_assets):
        for j in range(num_assets):
            for k in range(num_assets):
                coskew_matrix[i, j, k] = np.mean(
                    (returns.iloc[:, i] - means[i]) *
                    (returns.iloc[:, j] - means[j]) *
                    (returns.iloc[:, k] - means[k])
                ) / (std_devs[i] * std_devs[j] * std_devs[k])
    
    return coskew_matrix

# Calculate the Cokurtosis Matrix
def calculate_cokurt_matrix(returns):
    num_assets = returns.shape[1]
    cokurt_matrix = np.zeros((num_assets, num_assets, num_assets, num_assets))
    means = returns.mean(axis=0)
    std_devs = returns.std(axis=0)
    
    for i in range(num_assets):
        for j in range(num_assets):
            for k in range(num_assets):
                for l in range(num_assets):
                    cokurt_matrix[i, j, k, l] = np.mean(
                        (returns.iloc[:, i] - means[i]) *
                        (returns.iloc[:, j] - means[j]) *
                        (returns.iloc[:, k] - means[k]) *
                        (returns.iloc[:, l] - means[l])
                    ) / (std_devs[i] * std_devs[j] * std_devs[k] * std_devs[l])
    
    return cokurt_matrix

# asset_returns df has historical returns of the 8 ETFs
coskew_matrix = calculate_coskew_matrix(asset_returns)
cokurt_matrix = calculate_cokurt_matrix(asset_returns) 
 
# Perform optimization
risk_aversion = 3.07
optimal_weights_hom = optimize_portfolio(mu_implied, adjusted_covariance, coskew_matrix, cokurt_matrix, risk_aversion)

# Combine the optimized weights with the ETF names to display the final optimized portfolio 
optimal_weights_hom_df = pd.DataFrame({'ETF': etf_names, 'Optimized Weights': optimal_weights_hom})

# Display the optimized portfolio
print("Optimized Portfolio Weights:") 
print(optimal_weights_hom_df)
 

In [ ]:
# Function to calculate portfolio return, volatility, Sharpe ratio, coskewness, and cokurtosis
def calculate_portfolio_metrics(weights, returns, cov_matrix, rf, coskew_matrix, cokurt_matrix):
    # Calculate portfolio return (expected return)
    portfolio_return = np.dot(weights, returns)  # Use returns directly here, not the mean

    # Calculate portfolio volatility (risk)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

    # Calculate coskewness
    portfolio_coskewness = calculate_portfolio_coskewness(weights, coskew_matrix)

    # Calculate cokurtosis
    portfolio_cokurtosis = calculate_portfolio_cokurtosis(weights, cokurt_matrix)

    # Calculate Sharpe ratio (excluding higher moments for this calculation)
    sharpe_ratio = (portfolio_return - rf) / portfolio_volatility

    return portfolio_return, portfolio_volatility, sharpe_ratio, portfolio_coskewness, portfolio_cokurtosis


hom_return, hom_vol, hom_sharpe, hom_coskew, hom_cokurt = calculate_portfolio_metrics(
    optimal_weights, adjusted_returns, adjusted_covariance, annual_rfr, coskew_matrix, cokurt_matrix
) 

# Output the performance metrics
print(f"Higher Order Moments Portfolio Return: {hom_return:.2%}")
print(f"Higher Order Moments Portfolio Volatility: {hom_vol:.2%}")
print(f"Higher Order Moments Portfolio Sharpe Ratio: {hom_sharpe:.2}")
print(f"Higher Order Moments Portfolio Coskewness: {hom_coskew:.2}")
print(f"Higher Order Moments Portfolio Cokurtosis: {hom_cokurt:.2}")

**Objective Alignment:**

1. Coskewness: Minimizing coskewness focuses on reducing the likelihood of extreme negative returns, as portfolios with negative coskewness are more prone to downside risk. Our goal is to mitigate downside risk or avoid heavy left-tail risk (negative skewness), then the Min Coskewness Portfolio (w_min_coskewness) might be more appropriate.

2. Cokurtosis: Maximizing cokurtosis seeks to manage the portfolio's exposure to extreme events, or "fat tails," which occur in both directions (positive or negative). If we're concerned about tail risks in both directions and want to focus on the potential for positive extreme returns, then the Max Cokurtosis Portfolio (w_max_cokurtosis) could be a better fit.

**Risk Preferences:**

1. Risk-Averse Investors: If investors are more risk-averse and focused on protecting against downside risk, the Min Coskewness Portfolio would likely be more appropriate.

2. Return-Seeking Investors: If investors are more return-seeking and are willing to tolerate both upside and downside tail risks, the Max Cokurtosis Portfolio may offer better opportunities for higher returns in extreme market conditions.

Here we compare both portfolios' returns, volatility, and Sharpe ratio. 
If one portfolio has a higher Sharpe ratio (i.e., better risk-adjusted returns), that could be the better choice for w_hom.


### 4. Minimize Tracking Error 

Tracking error is the standard deviation of the difference between the portfolio returns and the benchmark returns. To minimize tracking error, we optimize the portfolio to minimize the deviation between the portfolio's returns and the returns of SPY benchmark. 

In [ ]:
# Check the benchmark data
spy_benchmark.tail(5)

In [ ]:
# Compute 'spy_returns' from the daily returns of SPY
spy_returns = spy_benchmark['Daily Returns']
spy_returns

In [ ]:
returns = returns.reindex(columns=etf_names)
returns

In [ ]:
returns, spy_returns = returns.align(spy_returns, join='inner', axis=0)
print('Check on length alignment:', len(spy_returns) == len(returns))

In [ ]:
def calculate_tracking_error(weights, portfolio_returns, benchmark_returns):
    """
    Calculate the tracking error between the portfolio and the benchmark.
    :param weights: Weights of the assets in the portfolio.
    :param portfolio_returns: Matrix of asset returns.
    :param benchmark_returns: Returns of the benchmark.
    :return: Tracking error (standard deviation of the difference).
    """
    portfolio_return = np.dot(portfolio_returns, weights)
    excess_returns = portfolio_return - benchmark_returns
    tracking_error = np.std(excess_returns)
    return tracking_error

def minimize_tracking_error(portfolio_returns, benchmark_returns):
    """
    Minimize the tracking error between a portfolio and a benchmark.
    :param portfolio_returns: Matrix of asset returns.
    :param benchmark_returns: Returns of the benchmark.
    :return: Optimal portfolio weights.
    """
    num_assets = portfolio_returns.shape[1]
    initial_weights = np.ones(num_assets) / num_assets  # Equal weight initialization
    
    # Bounds for the weights: no short-selling (weights between 0 and 1)
    bounds = [(0, 1) for _ in range(num_assets)]

    # Constraints for the optimization: fully invested portfolio
    constraints = {'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1} 

    # Perform the optimization
    result = minimize(calculate_tracking_error, initial_weights, args=(portfolio_returns, benchmark_returns),
        method='SLSQP', bounds=bounds, constraints=constraints)

    return result.x  # Optimal weights

# Assuming 'returns' is the matrix of asset returns and 'spy_returns' is the benchmark returns series
optimal_weights_tracking_error = minimize_tracking_error(returns.values, spy_returns.values) 

# Display the weights in a readable format
tracking_error_weights_df = pd.DataFrame({'ETF': returns.columns, 'Tracking Error Minimized Weight (%)': optimal_weights_tracking_error})

# New variable that includes the % symbol
formatted_weights = tracking_error_weights_df['ETF'] + ': ' + (tracking_error_weights_df['Tracking Error Minimized Weight (%)'] * 100).round(2).astype(str) + '%'

# Display the formatted weights in a readable format
#print(tracking_error_weights_df[['ETF', 'Tracking Error Minimized Weight (%)']].to_string(index=False))

print("Tracking Error Minimized Portfolio Weights:")
print(formatted_weights.to_string(index=False))

# Calculate metrics for the Tracking Error Minimized Portfolio
tracking_error_return = np.dot(returns.values, optimal_weights_tracking_error).mean()  # Expected return
tracking_error_vol = np.dot(returns.values, optimal_weights_tracking_error).std()  # Standard deviation (risk)
tracking_error_sharpe = (tracking_error_return - annual_rfr) / tracking_error_vol  # Sharpe ratio (assuming rfr is risk-free rate)
tracking_error_value = calculate_tracking_error(optimal_weights_tracking_error, returns.values, spy_returns.values)  # Tracking error

In [ ]:
# Output the performance metrics for Min Tracking Error
print(f"Min Tracking Error Portfolio Return: {tracking_error_return:.2%}")
print(f"Min Tracking Error Portfolio Volatility: {tracking_error_vol:.2%}")
print(f"Min Tracking Error Portfolio Sharpe Ratio: {tracking_error_sharpe:.2}")
print(f"Min Tracking Error Value: {tracking_error_value:.2%}")

In [ ]:
# Function to plot ETF allocations for tracking error
def plot_tracking_error_allocations(weights_df, title, etf_names):
    weights_percentage = weights_df['Tracking Error Minimized Weight (%)'] * 100
    
    plt.figure(figsize=(12, 6))
    plt.bar(etf_names, weights_percentage)
    plt.xlabel('ETF Names')
    plt.ylabel('Weight Allocation (%)')
    plt.title(f'{title} Portfolio Allocations')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Plot allocations for Tracking Error Minimized Portfolio
plot_tracking_error_allocations(tracking_error_weights_df, "Tracking Error Minimized", etf_names)

In [ ]:
metrics_df = pd.DataFrame({
    'Metric': ['Expected Return', 'Standard Deviation (Risk)', 'Sharpe Ratio', 'Tracking Error'],
    'Mean-Variance Portfolio': [f'{mv_return:.2%}', f'{mv_sharpe_vol:.2%}', f'{mv_sharpe_ratio:.2f}', 'N/A'],
    'Max Sharpe Portfolio': [f'{max_sharpe_return:.2%}', f'{max_sharpe_vol:.2%}', f'{max_sharpe_sharpe:.2f}', 'N/A'],
    'Higher-Order Moments Portfolio': [f'{hom_return:.2%}', f'{hom_vol:.2%}', f'{hom_sharpe:.2f}', 'N/A'],
    'Tracking Error Minimized Portfolio': [f'{tracking_error_return:.2%}', f'{tracking_error_vol:.2%}', f'{tracking_error_sharpe:.2f}', f'{tracking_error_value:.2%}']
})

# Display the metrics
print("\nPortfolio Metrics:")
metrics_df

In [ ]:
# Function to plot ETF allocations within a subplot
def plot_allocations_subplot(weights, title, etf_names, position):
    weights_percentage = weights * 100
    
    plt.subplot(2, 2, position)  # Adjust to 2x2 grid, position specified
    plt.bar(etf_names, weights_percentage)
    plt.xlabel('ETF Names')
    plt.ylabel('Weight Allocation (%)')
    plt.title(f'{title} Portfolio Allocations')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

# Prepare to create a figure with 4 subplots
plt.figure(figsize=(16, 12))

# Plot allocations for each portfolio in a 2x2 grid
plot_allocations_subplot(mv_weights, "Mean-Variance", etf_names, 1)
plot_allocations_subplot(w_max_sharpe, "Maximum Sharpe Ratio", etf_names, 2)
plot_allocations_subplot(optimal_weights_hom, "Higher-Order Moments", etf_names, 3) 
plot_allocations_subplot(optimal_weights_tracking_error, "Tracking Error Minimized", etf_names, 4)

# Display the combined plot
plt.show()

### Compute optimal allocations for three common levels of risk aversion (Trustee / Market / Kelly Investor)

In [ ]:
# Define risk aversion levels
risk_aversion_levels = {
    "Trustee": 3.07,  # Typically conservative
    "Market": 1,      # Market standard, usually around 1
    "Kelly": 0.5      # Aggressive, representing a Kelly criterion investor
}

# Function to compute allocations based on risk aversion
def adjust_for_risk_aversion(weights_df, weight_col, risk_aversion_levels):
    adjusted_weights = {}
    
    for level, risk_aversion in risk_aversion_levels.items():
        adjusted_col = f"{weight_col}_{level}_Adjusted"
        # Adjust the weights by dividing by the risk aversion level
        weights_df[adjusted_col] = weights_df[weight_col] / risk_aversion
        adjusted_weights[level] = weights_df[[adjusted_col]]
    
    return adjusted_weights

# Mean-Variance Adjusted Weights
mv_adjusted = adjust_for_risk_aversion(mv_weights_df, 'MV Weight', risk_aversion_levels)

# Max Sharpe Ratio Adjusted Weights
max_sharpe_adjusted = adjust_for_risk_aversion(max_sharpe_weights_df, 'Optimized Weights', risk_aversion_levels)

# Tracking Error Adjusted Weights
tracking_error_adjusted = adjust_for_risk_aversion(tracking_error_weights_df, 'Tracking Error Minimized Weight (%)', risk_aversion_levels)

# Higher-Order Moments Adjusted Weights
hom_adjusted = adjust_for_risk_aversion(optimal_weights_hom_df, 'Optimized Weights', risk_aversion_levels)

# Create DataFrame with all risk levels and optimizations
allocation_df = pd.DataFrame({
    'ETF': etf_names,
    'MV Trustee': mv_adjusted['Trustee'].squeeze(),  # Extract the column only
    'MV Market': mv_adjusted['Market'].squeeze(),
    'MV Kelly': mv_adjusted['Kelly'].squeeze(),
    'Max Sharpe Trustee': max_sharpe_adjusted['Trustee'].squeeze(),
    'Max Sharpe Market': max_sharpe_adjusted['Market'].squeeze(),
    'Max Sharpe Kelly': max_sharpe_adjusted['Kelly'].squeeze(),
    'Tracking Error Trustee': tracking_error_adjusted['Trustee'].squeeze(),
    'Tracking Error Market': tracking_error_adjusted['Market'].squeeze(),
    'Tracking Error Kelly': tracking_error_adjusted['Kelly'].squeeze(),
    'Higher-Order Moments Trustee': hom_adjusted['Trustee'].squeeze(),
    'Higher-Order Moments Market': hom_adjusted['Market'].squeeze(),
    'Higher-Order Moments Kelly': hom_adjusted['Kelly'].squeeze()
})

# Display the DataFrame
allocation_df


### Implementing Factor Analysis: A Fama-French Five-Factor Approach

**Data Preprocessing:**
To analyze the portfolio's factor exposures, we incorporate the **Fama-French Five-Factor Model**. The primary benchmark is the S&P 500 ETF, sourced from Yahoo Finance, serving as a proxy for the market index. These factors are as follows:

- **Market Risk Premium (Mkt-RF):** The return of the market above the risk-free rate.
- **Size (SMB):** The return difference between small-cap and large-cap stocks.
- **Value (HML):** The return spread between value and growth stocks.
- **Profitability (RMW):** The performance differential between firms with strong vs. weak profitability.
- **Investment (CMA):** The return spread between firms with conservative vs. aggressive investment strategies.

Both the ETF data and the Fama-French factor data are aligned on a **monthly frequency** for consistency.

**Factor Regression Analysis:**
We perform a regression of the portfolio's returns against these five factors to assess **factor sensitivities** (loadings). The factor loadings provide insight into the portfolio’s exposure to various market characteristics.

- A **positive factor loading** (e.g., on SMB) indicates a tilt towards small-cap stocks.
- A **negative factor loading** suggests the opposite, such as a preference for large-cap stocks.

The analysis allows us to explore several critical questions:
- Which factors contribute positively to the portfolio’s performance?
- Are certain factor exposures adding unintended risks?
- How do these factor exposures align with the investor’s overall strategy?

For instance, if the portfolio shows a **positive SMB loading**, indicating a strong tilt toward small-cap stocks, but the investor anticipates a downturn in this segment, it may be prudent to reduce exposure. Conversely, a positive outlook for value stocks (HML) could suggest increasing exposure to this factor.

**Portfolio Optimization:**
A key strategy for improving the portfolio is to adjust exposure to factors that either enhance returns or mitigate risk. For example, reducing exposure to small-cap stocks (SMB) and increasing exposure to large-cap stocks can be accomplished through re-weighting the portfolio.

**Advanced Approach: Long-Short Portfolio Construction:**
An alternative strategy involves constructing a **long-short portfolio**, going long on assets with positive exposure to desired factors and short on those with negative exposures. This allows for the neutralization of unwanted risks while amplifying returns from favored factors. However, this is currently out of this project scope.

**Portfolio Metrics and Adjustments:**
After implementing factor adjustments, key portfolio metrics are recalculated, including:
- **Expected Return**
- **Volatility (Risk)**
- **Sharpe Ratio**, which assesses the risk-adjusted return of the portfolio.

**Adjusting Portfolio Weights:**
Based on the factor analysis, specific adjustments to ETF allocations can be made to align with desired factor exposures, for example:

- **Reduce Exposure:**
   - **SMB (Size):** Decrease the allocation to 'VB' (small-cap) and potentially 'SPHD' if it is heavily weighted towards smaller, less profitable companies.
   - **HML (Value):** Reduce exposure to 'VTV' (value stocks) if value factors are expected to underperform.
   - **RMW (Profitability):** Decrease allocations to sectors with weaker profitability, such as 'XLE' (energy) or 'IEO' (oil and gas), which may be tied to lower RMW.

- **Increase Exposure:**
   - **CMA (Investment):** Increase allocations to ETFs with high exposure to conservative investment strategies. This could include 'SCHO' (short-term U.S. Treasury), 'TLT' (long-term U.S. Treasury), or 'XLK' (technology sector) if aligned with conservative capital expenditure patterns.

Through these strategic weight adjustments and optimizations, the portfolio can be fine-tuned to reflect the investor's views and market conditions while managing risk effectively. 

Fama French Five Factors monthly datasets retrieved from: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html

In [ ]:
# Load Fama French five factors dataset
factors = pd.read_csv('data/F-F_Research_Data_5_Factors_2x3_monthly.csv', index_col=0, parse_dates=True) # , skiprows=[0])

# Reset the index to work with DateCode if needed
factors.reset_index(inplace=True)

if factors.index.name == 'index':
    factors.reset_index(inplace=True)

# Convert the 'DateCode' to a readable format
factors['FormattedDate'] = pd.to_datetime(factors['index'], format='%Y%m').dt.strftime('%Y-%m-01')
factors.drop(columns=['index'], inplace=True)
factors.set_index('FormattedDate', inplace=True)
factors.index.name = 'Date'
 
#factors.index = pd.to_datetime(factors.index, format='%Y-%m-%d')

print(factors.head())
factors.tail()

In [ ]:
collect_df = collect_data(etf_names, '2019-12-01', '2024-07-01', interval='1mo', marketcap=False) # monthly period

In [ ]:
# Initialize an empty list to hold the DataFrames
temp = []

# Iterate through the dictionary and prepare each ETF DataFrame
for etf, df in collect_df.items():
    df['ETF'] = etf  # Add a column with the ETF name
    temp.append(df)

# Concatenate all DataFrames in the list into a single DataFrame
returns_df = pd.concat(temp)

# Display the resulting DataFrame
print(returns_df.head())
print(returns_df.tail())

In [ ]:
pd.set_option('display.max_rows', None)
display(returns_df)

In [ ]:
W_a

In [ ]:
# Regress each ETF's returns against the Fama-French factors
factor_exposures = {}
start_date = '2020-01-01'
end_date = '2024-06-01'
    
# Determine the date range from returns_df
print('Start:',start_date, 'End:', end_date)

# Filter the factors DataFrame to match the date range of returns_df
factors = factors[(factors.index >= start_date) & (factors.index <= end_date)]
print(factors.head())
factors.tail()

In [ ]:
# Convert the annual rate to a monthly rate
monthly_rfr = (1 + annual_rfr) ** (1/12) - 1

# Filter the returns_df to get 'Adj Close' prices for each ETF
prices_df = returns_df.pivot_table(index='Date', columns='ETF', values='Adj Close')

# Calculate daily returns
returns = prices_df.pct_change().dropna()
print(returns.head())
returns.tail()

In [ ]:
# Regress each ETF's returns against the Fama-French factors

# Calculate excess returns
#excess_returns = returns.subtract(monthly_rfr, axis=0) 

# Ensure the factors and returns data are aligned
factors.index = pd.to_datetime(factors.index, format='%Y-%m-%d')
returns = returns.reindex(factors.index).dropna()

factor_loadings = {}

# Set up the 4x4 subplot grid
fig, axs = plt.subplots(4, 2, figsize=(20, 20))  # 4x4 grid for the plots
fig.subplots_adjust(hspace=0.5, wspace=0.5)  # Adjust spacing between subplots

# Initialize subplot counter
plot_idx = 0

# Iterate over each ETF, perform regression, and plot
for etf in returns.columns:
    
    # Step 1: Align returns and factors
    y = returns[etf]
    X = factors[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]  # Use relevant factors
    rfr = factors['RF']  # Risk-free rate

    # Add constant to the factors for the regression intercept
    X = sm.add_constant(X)

    # Align excess returns (ETF returns - risk-free rate)
    y_aligned = y - rfr
    y_aligned = y_aligned.dropna()  # Drop missing values

    # Align factors with y_aligned
    X_aligned = X.loc[y_aligned.index]

    # Step 2: Perform regression (OLS)
    model = sm.OLS(y_aligned, X_aligned).fit()

    # Step 3: Plotting results
    ax = axs[plot_idx // 2, plot_idx % 2]  # 4x4 grid positioning

    # Scatter plot of excess ETF returns vs. market excess returns (Mkt-RF)
    ax.scatter(X_aligned['Mkt-RF'], y_aligned, alpha=0.5)

    # Calculate predicted values based only on the 'Mkt-RF' factor
    X_mkt_rf = sm.add_constant(X_aligned[['Mkt-RF']])  # Add constant term for intercept
    predicted_mkt_rf = model.params['const'] + model.params['Mkt-RF'] * X_aligned['Mkt-RF']

    # Plot the regression line based on 'Mkt-RF' only
    ax.plot(X_aligned['Mkt-RF'], predicted_mkt_rf, color='red', linewidth=2)
    ax.set_title(f'{etf} vs Mkt-RF', fontsize=10)
    ax.set_xlabel('Market Excess Returns (Mkt-RF)', fontsize=8)
    ax.set_ylabel(f'{etf} Excess Returns', fontsize=8)

    plot_idx += 1  # Move to the next subplot
    
    # Drop the intercept and ensure it is a Series
    factor_loadings[etf] = model.params.drop('const').squeeze()  

# Convert the factor loadings dictionary to a DataFrame
factor_loadings_df = pd.DataFrame.from_dict(factor_loadings, orient='index')

# Display the DataFrame in a readable format
print("\nFactor Loadings for Each ETF:")
print(factor_loadings_df)

### Strategy of Dynamic Weight Adjustment Based on Factor Loadings

In the Fama-MacBeth regression method, we reference it by dynamically adjusting exposures based on empirical factor loadings, providing a robust mechanism to optimize the portfolio according to the market's factor premiums:

1. Automate Factor Adjustment: The code should dynamically check if the loading is positive or negative and adjust accordingly without hardcoding the specific factors.

2. Incorporate Portfolio Constraints: Ensure that adjustments comply with any portfolio constraints, such as maximum or minimum weight limits.

Here are the 3 steps process:

A. Based on the factor loadings provided for each ETF, determine if they should be maximized or minimized:

1. Mkt-RF (Market Risk Premium): Generally maximize exposure if positive.

2. SMB (Small Minus Big): Minimize if to reduce small-cap exposure.

3. HML (High Minus Low): Minimize for growth-oriented portfolios, maximize for value.

4. RMW (Robust Minus Weak): Maximize to increase exposure to profitable firms.

5. CMA (Conservative Minus Aggressive): Maximize to increase exposure to conservative investment strategies.

B. Adjust Weights: For each ETF, adjust the weights based on the factor loadings:

1. Increase the weight if a factor needs to be maximized.

2. Decrease the weight if a factor needs to be minimized.

C. Normalization: After adjusting the weights, normalize them to ensure they sum to 100%. 


In [ ]:
# Flatten the factor loadings into a single list of absolute values
absolute_loadings = []
for etf, factors in factor_loadings.items():
    absolute_loadings.extend([abs(float(value)) for value in factors])

# Convert to a NumPy array for calculations
absolute_loadings = np.array(absolute_loadings)

# Calculate mean and standard deviation
mean_loading = np.mean(absolute_loadings)
std_loading = np.std(absolute_loadings)

# Set threshold as mean + 1 standard deviation
threshold = mean_loading + std_loading

print(f"Calculated Threshold: {threshold:.5f}")

# Apply the threshold to identify which loadings exceed the threshold
filtered_loadings_df = factor_loadings_df.applymap(lambda x: x if abs(x) >= threshold else np.nan)

# Display the filtered DataFrame in a readable format
print("\nFiltered Factor Loadings (Only those exceeding the threshold):")
filtered_loadings_df

**Evaluation Comments**

Using threshold at 0.02 resulted in a skew towards positive numbers and eliminating all negative loadings. Therefore there's a need to analyze the unique appearance of negative loadings and their values as follows: 

- -0.001 : 1 time
- -0.003 : 1 time
- -0.007 : 1 time
- -0.008 : 1 time
- -0.009 : 3 times
- -0.010 : 2 times
- -0.011 : 2 times
- -0.012 : 1 time
- -0.013 : 2 times

We observe from above that choosing a threshold value of -0.007 or -0.008 will result in a more balanced factor loadings table. 

For positive loadings:

Here's the list with the requested format:

- 0.008 : 5 times
- 0.009 : 6 times
- 0.010 : 4 times
- 0.011 : 2 times
- 0.012 : 3 times
- 0.013 : 4 times
- 0.014 : 1 time
- 0.017 : 3 times
- 0.018 : 3 times
- 0.019 : 5 times
- 0.020 : 5 times
- 0.021 : 6 times
- 0.022 : 4 times
- 0.023 : 4 times
- 0.024 : 6 times
- 0.025 : 3 times
- 0.026 : 1 time

We observe that a positive threshold of 0.010 could balance between making adjustments for common but not insignificant factors.

Let's filter the factor loadings dataframe with these two values and amend the code to allow positive and negative thresholds.

In [ ]:
pos_threshold = 0.010
neg_threshold = 0.008

# Apply the threshold to identify which loadings exceed the positive or negative thresholds
filtered_loadings_df = factor_loadings_df.applymap(
    lambda x: x if (x >= pos_threshold or x <= -neg_threshold) else np.nan
)

# Display the filtered DataFrame in a readable format
print("\nFiltered Factor Loadings (Only those exceeding the thresholds):")
filtered_loadings_df

The factor loadings for each ETF indicate how sensitive the ETF's returns are to each of the five Fama-French factors (Mkt-RF, SMB, HML, RMW, CMA). Here's what each of these loadings suggests about our W_a portfolio:

1. Mkt-RF (Market Risk Premium): A positive loading indicates that the ETF is sensitive to market movements. Most ETFs in this list show positive exposure, suggesting they generally move with the market.

2. SMB (Small Minus Big): Positive values suggest exposure to small-cap stocks, while negative values indicate large-cap exposure. For example, VB has a positive loading on SMB, indicating small-cap exposure.

3. HML (High Minus Low): Positive values suggest exposure to value stocks, while negative values indicate growth stock exposure. ETFs like SPHD and XLE show positive exposure, indicating a tilt towards value stocks.

4. RMW (Robust Minus Weak): This factor measures exposure to companies with robust profitability. Negative values, such as in XLE and IEO, suggest exposure to less profitable companies.

5. CMA (Conservative Minus Aggressive): Positive values indicate exposure to companies with conservative investment strategies, while negative values indicate aggressive strategies. For example, GLD shows a positive loading, indicating conservative exposure.

In [ ]:
# Initialize the adjusted weights dictionary
adjusted_weights = W_a.copy()

def adjust_weights_based_on_factors(allocations, factor_loadings, threshold=0.002):
    adjusted_weights = allocations.copy()
    print("Starting Adjustments...")

    for etf in factor_loadings.index:
        print(f"\nAdjustments for {etf}:")
        for factor, loading in factor_loadings.loc[etf].items():
            if pd.notna(loading) and abs(loading) >= threshold:
                if loading > 0:
                    print(f"Increasing exposure to {etf} due to positive {factor} factor.")
                    adjusted_weights[etf] *= 1.1  # Increase exposure by 10%
                elif loading < 0:
                    print(f"Decreasing exposure to {etf} due to negative {factor} factor.")
                    adjusted_weights[etf] *= 0.9  # Decrease exposure by 10%
            else:
                print(f"No significant adjustment for {etf} due to {factor} factor.")

    # Normalize weights to sum to 100%
    total_weight = adjusted_weights.sum()
    adjusted_weights /= total_weight

    return adjusted_weights


# Adjust weights dynamically based on the factor loadings
adjusted_weights = adjust_weights_based_on_factors(W_a, filtered_loadings_df)

# Display the adjusted weights
adjusted_weights_df = pd.DataFrame({'ETF': adjusted_weights.index, 'Previous Weight W_a': W_a.values, 'Adjusted Weight': adjusted_weights.values})

adjusted_weights_df

In [ ]:
# Full Portfolio Metrics Calculation 

# `adjusted_weights` is the result after adjusting allocations based on factor loadings
portfolio_weights = adjusted_weights

# Call the function to calculate R and C from monthly filtered ETFs' returns
R, C = assets_historical_returns_and_covariances(returns)

# Display the results
df_R = pd.DataFrame(R, columns=["Annualized Expected Returns"])

# Print the DataFrame nicely
print(df_R)

print("\nAnnualized Covariance Matrix:") 

# Convert the NumPy array to a DataFrame with ETF names as index and columns
df_C = pd.DataFrame(C, index=etf_names, columns=etf_names)
 
# Style the DataFrame for a nicer display
df_C = df_C.round(3).style.set_table_styles(
    [{'selector': 'th', 'props': [('font-size', '7.8pt')]},
     {'selector': 'td', 'props': [('font-size', '7.8pt')]}]
)

# Display the styled DataFrame
display(df_C)

In [ ]:
# Calculate the portfolio metrics using the adjusted weights
portfolio_return, portfolio_vol, portfolio_sharpe = portfolio_metrics(portfolio_weights, R, C, annual_rfr)

# Create a dictionary to store the metrics
metrics = {
    "Metric": ["Expected Return", "Volatility (Risk)", "Sharpe Ratio"],
    "Value": [f"{portfolio_return:.2%}", f"{portfolio_vol:.2%}", f"{portfolio_sharpe:.2f}"]
}

# Convert the dictionary into a DataFrame
df_metrics = pd.DataFrame(metrics)

# Print the DataFrame nicely
print("\nPortfolio Metrics with Optimized Weights:")
df_metrics

In [ ]:
portfolio_weights

### 4. Betting against Beta Custom Factor

Betting Against Beta is a strategy that goes long on low-beta stocks and shorts high-beta stocks, betting that low-beta stocks will outperform on a risk-adjusted basis. The steps are:

1. Identify High-Beta and Low-Beta ETFs: Based on the betas of the 20 ETFs, we classify ETFs into high-neta and low-beta groups.

2. Construct a BAB Portfolio: Construct a portfolio that is long low-beta ETFs and short high-beta ETFs.

3. Calculate the Returns for BAB: Compute the excess returns for the BAB portfolio over time and use it as a factor for backtesting.

In [ ]:
# Separate ETFs into high-beta and low-beta based on the calculated regression betas
high_beta_threshold = betas_df['Regression Beta'].quantile(0.75)  # Top 25% are high-beta
low_beta_threshold = betas_df['Regression Beta'].quantile(0.25)  # Bottom 25% are low-beta

# Create lists for high-beta and low-beta ETFs
high_beta_etfs = betas_df[betas_df['Regression Beta'] > high_beta_threshold].index.tolist()
low_beta_etfs = betas_df[betas_df['Regression Beta'] < low_beta_threshold].index.tolist()

print('High Beta ETFs:')
display(high_beta_etfs)

print('Low Beta ETFs:')
display(low_beta_etfs)

In [ ]:
# Calculate the returns for high-beta and low-beta portfolios
high_beta_returns = asset_returns[high_beta_etfs].mean(axis=1)   
low_beta_returns = asset_returns[low_beta_etfs].mean(axis=1)  

# Compute the BAB (Betting Against Beta) factor returns: Long low-beta, Short high-beta
bab_returns = low_beta_returns - high_beta_returns

# Display BAB returns
print("BAB Factor Returns:")
print(bab_returns.head())

In this section, we compute the rolling beta of the BAB factor relative to SPY, calculated over a moving window. The BAB factor, constructed as the difference between low-beta and high-beta ETF returns, has its own dynamic relationship with the market.

The purpose of this regression is to track how the BAB factor's sensitivity to the market evolves over time. Unlike the individual ETF betas (which are static), the rolling beta gives  a time-varying measure of the relationship between BAB and SPY, which can change as market conditions fluctuate.

In [ ]:
# Create a DataFrame to store market (SPY) returns and BAB factor returns for comparison
combined_returns = pd.DataFrame({
    'SPY Returns': spy_returns,
    'BAB Returns': bab_returns
}).dropna()

# Define a rolling window for calculating rolling beta and alpha
rolling_window = best_window

# Initialize lists to store rolling beta and alpha
rolling_betas = []
rolling_alphas = []

# Loop through the combined returns and calculate rolling beta and alpha
for i in range(rolling_window, len(combined_returns)):
    window_data = combined_returns.iloc[i - rolling_window:i]
    X = sm.add_constant(window_data['SPY Returns'])
    y = window_data['BAB Returns']
    model = sm.OLS(y, X).fit()
    
    rolling_betas.append(model.params[1])  # Store the rolling beta
    rolling_alphas.append(model.params[0])  # Store the rolling alpha

# Create a DataFrame for rolling beta and alpha
rolling_metrics = pd.DataFrame({
    'Rolling Beta': rolling_betas,
    'Rolling Alpha': rolling_alphas
}, index=combined_returns.index[rolling_window:])

# Plot the rolling beta and alpha
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(rolling_metrics['Rolling Beta'], label='Rolling Beta (BAB vs SPY)', color='blue')
plt.title('Rolling Beta of BAB vs SPY')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(rolling_metrics['Rolling Alpha'], label='Rolling Alpha (BAB vs SPY)', color='green')
plt.title('Rolling Alpha of BAB vs SPY')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Calculate performance metrics for BAB and SPY
bab_annual_return = bab_returns.mean() * 252  # Assuming 252 trading days
spy_annual_return = spy_returns.mean() * 252

bab_volatility = bab_returns.std() * np.sqrt(252)
spy_volatility = spy_returns.std() * np.sqrt(252)

bab_sharpe_ratio = (bab_annual_return - annual_rfr) / bab_volatility
spy_sharpe_ratio = (spy_annual_return - annual_rfr) / spy_volatility

# Display the performance comparison
print(f"BAB Annual Return: {bab_annual_return:.2%}, Volatility: {bab_volatility:.2%}, Sharpe Ratio: {bab_sharpe_ratio:.2f}")
print(f"SPY Annual Return: {spy_annual_return:.2%}, Volatility: {spy_volatility:.2%}, Sharpe Ratio: {spy_sharpe_ratio:.2f}")

**Evaluation Comments**

Rolling Beta of BAB vs SPY:

1. The beta of the Betting Against Beta (BAB) strategy relative to SPY is consistently negative throughout the entire period, fluctuating between -1.4 and -2.2.

2. A negative beta indicates that the BAB strategy is inversely correlated with the market. When the market (SPY) rises, the BAB strategy tends to decline, and vice versa.

3. Notably, the beta becomes more negative during certain periods (e.g., late 2020 and early 2024), indicating that the BAB strategy’s inverse relationship with the market has strengthened during these times.

4. In mid-2024, there's a sharp downward spike in beta, potentially signifying a period of high market volatility or a significant shift in how high- and low-beta assets are behaving.

Rolling Alpha of BAB vs SPY:

1. The alpha of the BAB strategy fluctuates around zero for most of the period, with both positive and negative phases. However, the alpha remains quite low throughout, suggesting that the BAB strategy does not consistently generate strong excess returns.

2. Alpha represents the excess return that the BAB strategy generates over what would be expected based on its beta (market exposure). Positive alpha implies outperformance relative to the market.

3. From early 2021 until mid-2022, the alpha stays slightly negative, suggesting underperformance compared to expectations.

4. Around mid-2023, the rolling alpha shows signs of improvement, and there's a noticeable upward trend, indicating that the BAB strategy started to produce positive excess returns relative to SPY.

Performance Metrics for BAB vs SPY:

1. Negative annual return of the BAB strategy at -18.64% indicates poor performance during the analysis period. Given that the BAB strategy is built on long low-beta and short high-beta positions, the negative performance likely reflects a period where high-beta stocks outperformed low-beta stocks, which would hurt the BAB portfolio.

2. The BAB strategy is highly risky and volatile, with volatility higher than SPY.

3. The negative Sharpe ratio confirms that the BAB strategy delivered negative risk-adjusted returns. 

4. SPY delivered a solid positive return, positive Sharpe ratio, and lower volatility compared to BAB, which all in all indicating a strong bull market during the backtesting period from 2020-2024. 

Insights:

1. Perhaps one of the main issues with applying the BAB strategy to ETFs is that most ETFs are market-cap weighted. This means that the largest, most liquid stocks dominate the portfolio. As a result, the BAB strategy, which typically works by taking long positions in low-beta stocks and short positions in high-beta stocks, is less effective. High-beta stocks, especially small-cap or volatile stocks, are underrepresented in these ETFs, making it harder for the BAB strategy to produce the expected returns.

2. According to research, BAB's effectiveness comes from its ability to target small, high-beta stocks, which tend to underperform on a risk-adjusted basis. However, ETFs tend to focus on large-cap stocks, and this dilutes the BAB factor because it relies on the price movements of smaller, more volatile stocks that ETFs may not sufficiently capture.

3. BAB strategies may not work well in an ETF context because high-beta stocks tend to outperform in bull markets. Since the backtesting period from 2020-2024 is a bull market, making the short side of the BAB trade unprofitable.

### Kelly Criterion Portfolio Strategy

The objective here is to calculate the Kelly weights based on expected returns and covariance matrix and simulate the portfolio's performance over time, considering different fractions of Kelly capital allocation.

Kelly Allocation Calculation: The Kelly allocation formula (kelly_allocation) calculates the optimal weights for each asset using the expected returns and the covariance matrix. These weights are used in the kelly_strategy function.

Portfolio Performance Simulation: The kelly_strategy function now uses the Kelly weights to simulate portfolio performance. The portfolio return is computed as the weighted sum of individual asset returns at each time step.

Fractional Kelly: The strategy allows to scale down the Kelly allocation by a fraction (e.g., 0.5, 0.66) to control risk.

The plot visualizes the cumulative equity values for different fractions of the Kelly allocation over time, to compare performance based on different levels of risk aversion (i.e., full Kelly vs. scaled Kelly).

Reference: Yves Hilpisch’s Python for Algorithmic Trading book by O'Reilly


In [ ]:
# Kelly Criterion Portfolio Weights Calculation
def kelly_allocation(returns, cov_matrix, risk_free_rate=0.03):
    """
    Compute portfolio weights using the Kelly criterion.

    Parameters:
    returns: pd.Series - Expected returns for each asset (E[R])
    cov_matrix: pd.DataFrame - Covariance matrix of asset returns
    risk_free_rate: float - Risk-free rate used in excess returns (default: 3%)

    Returns:
    pd.Series - Portfolio weights according to the Kelly criterion
    """
    # Convert risk-free rate to excess returns
    excess_returns = returns - risk_free_rate
    
    # Reshape the excess_returns to a 2D column vector
    excess_returns = excess_returns.values.reshape(-1, 1)

    # Inverse of the covariance matrix
    inv_cov_matrix = np.linalg.inv(cov_matrix)

    # Kelly optimal weights: W = inv(Cov) * (E[R] - RiskFreeRate)
    kelly_weights = inv_cov_matrix @ excess_returns
    
    # Normalize weights to sum to 1 (optional)
    kelly_weights = kelly_weights.flatten()  # Convert back to 1D array
    kelly_weights /= np.sum(np.abs(kelly_weights))  # L1 norm ensures weights are properly scaled

    return pd.Series(kelly_weights, index=returns.index)

# Kelly Strategy Simulation
def kelly_strategy(data, f, weights):
    global equs  # List to store equity-related column names for plotting
    
    # Create column names for equity and capital based on the fraction (f)
    equ = 'equity_{:.2f}'.format(f)
    equs.append(equ)
    cap = 'capital_{:.2f}'.format(f)
    
    # Initialize the equity and capital columns
    equity_data[equ] = 1
    equity_data[cap] = equity_data[equ] * f
    
    for i, t in enumerate(data.index[1:]):
        t_1 = data.index[i]
        
        # Calculate portfolio return based on the weighted sum of asset returns
        portfolio_return = np.dot(data.iloc[i][list(weights.index)], weights)
        
        # Update capital based on the previous capital and the weighted portfolio return
        equity_data.loc[t, cap] = equity_data[cap].loc[t_1] * math.exp(portfolio_return)

        # Update equity and capital
        equity_data.loc[t, equ] = equity_data[cap].loc[t] - equity_data[cap].loc[t_1] + equity_data[equ].loc[t_1]
        equity_data.loc[t, cap] = equity_data[equ].loc[t] * f

portfolio_returns = returns.copy()        
C = portfolio_returns.cov()
print('C=', C)
        
# Calculate Kelly optimal weights for portfolio
kelly_weights = kelly_allocation(returns.mean(), C, annual_rfr)
 
# Initialize equity-related columns
equs = []

# Create a new DataFrame to store equity and capital values
equity_data = pd.DataFrame(index=returns.index)

# Apply the Kelly strategy for different fractions (scaled by 0.5, 0.66, and 1)
kelly_strategy(returns, f=0.5, weights=kelly_weights)
kelly_strategy(returns, f=0.66, weights=kelly_weights)
kelly_strategy(returns, f=1.0, weights=kelly_weights)

# Plot cumulative returns and equity values for each fraction
ax = returns.cumsum().apply(np.exp).plot(legend=True, figsize=(10, 6), label='Cumulative Returns')
equity_data[equs].plot(ax=ax, legend=True)

plt.title('Kelly Criterion Portfolio Performance with Different Fractions')
plt.ylabel('Cumulative Equity Value')
plt.xlabel('Date')
plt.show()


In [ ]:
equity_data

In [ ]:
# Filter the DataFrame for only the equity columns
equity_columns = ['equity_0.50', 'equity_0.66', 'equity_1.00']

# Plot only the equity values for the 0.5, 0.66, and 1.0 Kelly allocations
equity_data[equity_columns].plot(figsize=(10, 6), legend=True)

# Customize the plot
plt.title('Kelly Criterion Portfolio Performance (Equity Values)')
plt.ylabel('Cumulative Equity Value')
plt.xlabel('Date')

# Display the plot
plt.show()


In [ ]:
import matplotlib.colors as mcolors
 
colors = list(mcolors.TABLEAU_COLORS) + list(mcolors.CSS4_COLORS)[:8]  # Use a combination of TABLEAU and CSS4 colors

# Plot cumulative returns with distinct color
ax = returns.cumsum().apply(np.exp).plot(legend=True, figsize=(10, 6), color=colors[0], label='Cumulative Returns')

# Plot equity values with distinct colors for each fraction
equity_data[equs].plot(ax=ax, legend=True, color=colors[1:len(equs)+1])

# Customize the plot
plt.title('Kelly Criterion Portfolio Performance with Different Fractions')
plt.ylabel('Cumulative Equity Value')
plt.xlabel('Date')

# Display the plot
plt.show()

**Evaluation Comments**

The plot on performance of Kelly portfolios vs. ETFs where the three Kelly allocations (0.5, 0.66, and 1.0) exhibit almost flat performance over time, with very limited growth compared to the ETFs, which display much higher cumulative returns. 

the scaling of the Kelly allocation (fractional vs full) had little impact on the portfolios’ performance, and the portfolios were conservative and did not capture much upside in the market. The conservative returns could be due to the ‘returns.mean()’ being low and used in the Kelly allocation calculation, and normalization of the weights using the L1 norm led to conservative weights as it reduces exposure to more volatile assets. 